# Combat System Design

## Overview

Combat extends the FoodSimulator pattern: a turn-based simulator with callbacks, event logging, and integration with the existing piece stats and movement budget system. The key differentiator is **high ground**—elevation grants significant combat advantage.

---

## Core Mechanics

esults from `HexMagic/game/combat_balance.py` — a turn-based battle simulator for balancing piece stats.

## Design Goals

- **Rook**: High defense, low offense (tank)
- **Knight**: High offense, low defense (glass cannon)
- **Queen**: Balanced powerhouse
- **Target**: 1 Queen ≈ 1 Rook + 2 Knights

## Final Tuned Stats

| Piece  | Offense | Defense | Health | Role |
|--------|---------|---------|--------|------|
| QUEEN  | 8.4     | 1.29    | 121    | Powerhouse |
| ROOK   | 2.5     | 1.6     | 130    | Tank |
| KNIGHT | 7.5     | 0.5     | 55     | Glass cannon |
| BISHOP | 4.0     | 1.0     | 100    | Balanced |
| PAWN   | 1.5     | 0.8     | 80     | Worker |
| KING   | 3.0     | 1.5     | 100    | Protected |

## Battle Results (500 runs, 15% damage variance)

| Matchup | Winner | Winrate | Avg Turns |
|---------|--------|---------|-----------|
| Q vs R+2K | Queen | **60.4%** | 32.7 |
| Q vs R | Queen | 100% | 25.3 |
| Q vs 2K | Queen | 100% | 7.9 |
| R vs 2K | Knights | 100% | 17.7 |
| R vs K | Rook | 100% | 11.5 |
| Q vs 2R | Rooks | 100% | 38.7 |

## Key Findings

1. **Q vs R+2K at ~60%** — Close to the 50% target. The Queen has a slight edge, which feels right for a "powerhouse" unit.

2. **Knights punish Rooks** — 2 Knights beat a Rook 100% of the time. Glass cannons melt the tank before it can deal enough damage back.

3. **Rook beats single Knight** — In 1v1, the Rook's tankiness outlasts the Knight's burst.

4. **2 Rooks beat Queen** — Two tanks working together overwhelm the Queen.

5. **Damage variance matters** — Without variance, outcomes are deterministic. The 15% variance creates interesting probabilistic outcomes.

## Combat Mechanics

### Damage Formula
```
actual_damage = raw_offense / target_defense
```

### Turn Structure
1. All Team 0 units attack (simultaneously)
2. All Team 1 units attack (simultaneously)
3. Check for victory
4. Repeat

### Target Priority
Default: `lowest_hp` (focus fire on weakest enemy)

Options: `random`, `highest_offense`

## Usage

```python
from HexMagic.game.combat_balance import (
    test_default_balance,
    run_scenario_batch,
    scenario_custom,
    run_battle,
    COMBAT_PROFILES
)

# Run all standard scenarios
test_default_balance(runs=100)

# Custom matchup
summary = run_scenario_batch(
    ['QUEEN', 'PAWN'], 
    ['ROOK', 'KNIGHT', 'KNIGHT'],
    runs=100
)
print(f"Team0 wins: {summary.team0_winrate:.1%}")

# Single battle with custom variance
team0, team1 = scenario_custom(['QUEEN'], ['ROOK', 'KNIGHT'])
result = run_battle(team0, team1, damage_variance=0.2)
print(f"Winner: Team {result.winner}, Turns: {result.turns}")
```

## Tuning Notes

The balance is **sensitive** to small stat changes. During tuning:
- Queen offense 8.0 → 0% vs R+2K
- Queen offense 8.5 → 92% vs R+2K
- Queen offense 8.4 → 60% vs R+2K (sweet spot)

This sharp threshold means the "1Q = 1R + 2K" balance requires precise tuning. The `grid_search_balance()` function can help find optimal values if you want different balance points.

### Combat Range

Base combat range depends on piece type, modified by elevation:

```
combat_range = base_range + floor(elevation / elevation_delta)
```

Where `elevation_delta` is the terrain's elevation step (typically ~100m). A piece at elevation 500 on a map with 100m deltas gets +5 hex range. This makes hilltops and ridgelines strategically vital.

**Base ranges by type:**
- PAWN: 1 (melee worker, can defend but shouldn't fight)
- KNIGHT: 2 (fast scout, hit-and-run)
- ROOK: 3 (siege unit, stationary firepower)
- BISHOP: 2 (balanced, mobile support)
- QUEEN: 3 (powerful all-rounder)
- KING: 1 (moderate, stays protected)

### Damage Calculation

Damage dealt per attack:

```python
base_damage = attacker.attack_strength * attacker.size / 100
terrain_mod = 1.0 + (attacker_elevation - defender_elevation) * 0.1
size_ratio = min(1.5, attacker.size / max(1, defender.size))
damage = base_damage * terrain_mod * size_ratio
```

**PIECE_DEFAULTS reminder:** (attack, move, harvest, scout)
- PAWN: (1, 3, 7, 2) — weak attack
- KNIGHT: (2, 7, 1, 5) — scout, moderate attack
- ROOK: (7, 2, 1, 1) — **strongest attack**, slow
- BISHOP: (3, 4, 4, 4) — balanced
- QUEEN: (6, 5, 3, 5) — second strongest
- KING: (4, 2, 2, 3) — moderate defense

### High Ground Advantage

Elevation provides three bonuses:

1. **Extended Range**: +1 hex per elevation_delta (multiplicative with sight)
2. **Damage Bonus**: +10% per elevation_delta above target
3. **Defense Bonus**: Harder to hit from below (attackers at -10% per delta)

A unit on a 3-delta hill attacking down gets +30% damage and 3 extra range hexes.

### Resolution Order

Combat resolves by `RANK_ORDER` (King → Queen → Rook → Bishop → Knight → Pawn), then by food (hungrier pieces act first—desperation):

```python
RANK_ORDER = {
    PieceType.KING:   0,  # acts first (priority target, protected)
    PieceType.QUEEN:  1,
    PieceType.ROOK:   2,
    PieceType.BISHOP: 3,
    PieceType.KNIGHT: 4,
    PieceType.PAWN:   5,  # acts last
}

def combat_order(pieces: list[Piece]) -> list[Piece]:
    return sorted(pieces, key=lambda p: (
        RANK_ORDER.get(p.piece_type, 99), 
        p.food,  # hungrier acts first
        p.id     # tiebreaker
    ))
```

---

## CombatSimulator Design

Extends FoodSimulator's callback-driven architecture:

```python
@dataclass
class CombatEvent:
    turn: int
    slot: int
    event_type: CombatEventType  # ATTACKED, DEFENDED, RETREATED, KILLED, etc.
    attacker_id: str
    defender_id: str
    damage: float
    attacker_hex: int
    defender_hex: int
    elevation_diff: int

CombatCallback = Callable[[CombatEvent], None]

@dataclass
class CombatSimulator:
    """Turn-based combat with interleaved food/combat phases."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list[Piece]
    countries: np.ndarray = None
    elevation_mult: float = 0.005
    elevation_delta: float = 100.0  # meters per combat range bonus
    
    squads: list[Squad] = field(default_factory=list)
    callbacks: dict = field(default_factory=dict)  # squad.id → CombatCallback
    
    _rows: list = field(default_factory=list)
    turn: int = 0
```

### Event Types

```python
class CombatEventType(Enum):
    # Existing from SimEventType
    MOVED     = "moved"
    BLOCKED   = "blocked"
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"
    
    # Combat-specific
    ATTACKED   = "attacked"      # dealt damage
    DEFENDED   = "defended"      # took damage but survived
    ROUTED     = "routed"        # forced retreat
    KILLED     = "killed"        # reduced to 0 health
    CAPTURED   = "captured"      # special: King taken
    FLANKED    = "flanked"       # attacked from non-facing direction
```

### Turn Structure

Each turn follows this sequence:

```
1. Movement phase (existing)
   - Resolve by RANK_ORDER
   - Consume move budget
   
2. Combat phase (new)
   - Build target lists (enemies in range)
   - Resolve by RANK_ORDER
   - Apply damage, check deaths
   - Fire callbacks
   
3. Supply phase (existing)
   - Harvest
   - Give
   - Eat
   - Check starvation deaths
   
4. End-of-turn callbacks
```

### Combat Resolution

```python
def _combat_phase(self):
    """Resolve all combat for the current turn."""
    attackers = combat_order([p for p in self.alive if p.health > 0])
    
    for attacker in attackers:
        if attacker.health <= 0:
            continue  # died earlier this phase
            
        targets = self._find_targets(attacker)
        if not targets:
            continue
            
        # Target selection: lowest health enemy in range
        target = min(targets, key=lambda p: (p.health, p.id))
        damage = self._calc_damage(attacker, target)
        
        target.health -= damage
        self._log_combat(attacker, target, damage)
        
        if target.health <= 0:
            target.health = 0
            self._log_combat(attacker, target, 0, CombatEventType.KILLED)

def _find_targets(self, piece: Piece) -> list[Piece]:
    """Find enemy pieces within combat range."""
    piece_elev = self.elevations[piece.location]
    bonus_range = int(piece_elev / self.elevation_delta)
    total_range = BASE_COMBAT_RANGE[piece.piece_type] + bonus_range
    
    in_range = set()
    self._flood_range(piece.location, total_range, in_range)
    
    return [p for p in self.alive 
            if p.owner_id != piece.owner_id 
            and p.location in in_range]

def _calc_damage(self, attacker: Piece, defender: Piece) -> float:
    """Calculate damage with elevation modifiers."""
    atk_elev = self.elevations[attacker.location]
    def_elev = self.elevations[defender.location]
    elev_diff = (atk_elev - def_elev) / self.elevation_delta
    
    base = attacker.attack_strength * attacker.size / 100
    terrain_mod = 1.0 + elev_diff * 0.1
    size_ratio = min(1.5, attacker.size / max(1, defender.size))
    
    return max(1.0, base * terrain_mod * size_ratio)
```

---

## DEFEND Instruction

The existing `Instruction.DEFEND` (value 1) gains combat meaning:

```python
def _is_defending(self, piece: Piece) -> bool:
    """Check if piece is in DEFEND stance this turn."""
    il = piece.instructions
    if not il.rules:
        return False
    return Instruction(il.rules[il.cursor]) == Instruction.DEFEND

DEFEND_BONUS = 0.5  # take 50% less damage when defending
```

When defending:
- Movement budget is consumed (costs 1 slot)
- Piece does not move
- Incoming damage reduced by 50%
- +1 to combat range (fortified position)

---

## Flank Attacks

Attacking from outside a piece's facing direction grants bonus damage:

```python
def _is_flanking(self, attacker: Piece, defender: Piece) -> bool:
    """True if attacker is outside defender's forward arc."""
    # Defender's forward arc: facing ± 1 direction
    forward_arc = {defender.facing, (defender.facing + 1) % 6, (defender.facing - 1) % 6}
    
    # Direction from defender to attacker
    direction = self._hex_direction(defender.location, attacker.location)
    return direction not in forward_arc

FLANK_BONUS = 1.3  # +30% damage from flanks/rear
```

This makes facing and scouting strategically important—KNIGHT's high scout stat helps detect ambushes.

---

## Retreat & Rout

When a piece takes critical damage (below 25% health), it may rout:

```python
ROUT_THRESHOLD = 0.25  # 25% health

def _check_rout(self, piece: Piece, damage: float) -> bool:
    """Check if piece routs after taking damage."""
    if piece.health / piece.max_health > ROUT_THRESHOLD:
        return False
    
    # Rout chance based on remaining health and piece type
    base_chance = 0.5 * (1 - piece.health / piece.max_health)
    
    # Higher rank = less likely to rout
    rank_mod = RANK_ORDER.get(piece.piece_type, 5) * 0.1
    
    return random.random() < base_chance + rank_mod
```

Routed pieces:
- Automatically move away from nearest enemy
- Cannot attack this turn
- Recover composure next turn if not attacked

---

## Pyomo Integration: Tactical Optimization

Just like the supply chain optimization, combat can leverage Pyomo for tactical decisions:

### Problem: Optimal Attack Assignment

Given N attackers and M defenders, minimize total damage to friendlies while maximizing damage to enemies:

```python
def build_combat_model(attackers, defenders, grid, elevations, elevation_delta):
    """
    Sets:
        A = attackers
        D = defenders
        
    Params:
        damage[a,d] = predicted damage if a attacks d
        range_ok[a,d] = 1 if d is in range of a
        
    Vars:
        attack[a,d] ∈ {0,1} = 1 if a attacks d
        
    Constraints:
        ∀a: Σ_d attack[a,d] ≤ 1  # each attacker picks one target
        attack[a,d] ≤ range_ok[a,d]  # can only attack in range
        
    Objective:
        max Σ_{a,d} damage[a,d] * attack[a,d] * priority[d]
        
    Where priority[d] favors:
        - Low-health enemies (finish them off)
        - High-rank enemies (target queens/kings)
        - Enemies threatening our high-rank pieces
    """
    m = pyo.ConcreteModel()
    m.A = pyo.Set(initialize=[a.id for a in attackers])
    m.D = pyo.Set(initialize=[d.id for d in defenders])
    
    # Precompute damage and range
    damage_matrix = {}
    range_matrix = {}
    for a in attackers:
        for d in defenders:
            range_matrix[a.id, d.id] = _in_range(a, d, grid, elevations, elevation_delta)
            damage_matrix[a.id, d.id] = _calc_damage(a, d, elevations, elevation_delta)
    
    m.damage = pyo.Param(m.A, m.D, initialize=damage_matrix)
    m.range_ok = pyo.Param(m.A, m.D, initialize=range_matrix)
    
    m.attack = pyo.Var(m.A, m.D, domain=pyo.Binary)
    
    # One target per attacker
    def one_target(m, a):
        return sum(m.attack[a, d] for d in m.D) <= 1
    m.one_target = pyo.Constraint(m.A, rule=one_target)
    
    # Range constraint
    def in_range(m, a, d):
        return m.attack[a, d] <= m.range_ok[a, d]
    m.in_range = pyo.Constraint(m.A, m.D, rule=in_range)
    
    # Objective: maximize expected damage dealt
    def objective(m):
        return sum(m.damage[a, d] * m.attack[a, d] 
                   for a in m.A for d in m.D)
    m.obj = pyo.Objective(rule=objective, sense=pyo.maximize)
    
    return m
```

### Problem: Defensive Positioning

Where should pieces move to maximize defensive coverage of high-value targets (links to `warehouselocation.md`):

```python
# Inputs:
#   - High-value hexes (settlements, resources, chokepoints)
#   - Current enemy positions
#   - Terrain elevations

# Objective:
#   max(coverage of high-value hexes) + elevation_bonus - exposure_to_enemy
```

---

## Callbacks & Replanning

Following the FoodSimulator pattern, combat supports callbacks for adaptive AI:

```python
@dataclass
class CombatReport:
    """End-of-combat-phase context for callbacks."""
    turn: int
    squad: Squad
    enemies_killed: list[str]
    friendlies_killed: list[str]
    damage_dealt: float
    damage_taken: float
    routed: list[str]
    
def combat_callback_example(report: CombatReport):
    """Example: retreat if losing badly."""
    if report.damage_taken > report.damage_dealt * 2:
        # Trigger retreat orders
        for piece in report.squad.alive:
            piece.instructions = InstructionList(
                [Instruction.ROT_L.value] * 3 + [Instruction.FORWARD.value] * 5,
                patrol=False
            )
```

The Mission system can use combat callbacks to trigger replanning:

```python
class Mission:
    def combat_callback(self) -> CombatCallback:
        def cb(report: CombatReport):
            if report.friendlies_killed:
                self.replan()  # rebuild supply/combat orders
        return cb
```

---

## Integration with Existing Systems

### FoodSimulator Extension

CombatSimulator inherits from FoodSimulator, adding the combat phase:

```python
class CombatSimulator(FoodSimulator):
    def run(self, turns: int):
        for _ in range(turns):
            self._movement_phase()
            self._combat_phase()      # NEW
            self._harvest_phase()
            self._give_phase()
            self._eat_phase()
            self._end_of_turn_callbacks()
            self.turn += 1
```

### PieceBoard Testing

Extend PieceBoard for combat testing scenarios:

```python
class CombatBoard(PieceBoard):
    """Test board with enemy pieces for combat scenarios."""
    
    def add_enemy_squad(self, count: int, positions: list[int]):
        """Add enemy pieces at specified positions."""
        ...
    
    def run_combat_sim(self, turns: int) -> pd.DataFrame:
        """Run combat simulation, return event log."""
        ...
```

In [ ]:
#| default_exp game/mechanics/simulator

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import FlowOverlay, CreamOverlay, RiverOverlay, ClimateOverlay, OverlayContext
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.plot.cube import HexPosition


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc

import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass, field
from functools import cached_property
#| export
import heapq
import io
from monsterui.all import *
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging

from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.flag import GameContext
from HexMagic.game.data import RANK_ORDER 

from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from HexMagic.game.piece import SquadPlanOverlay, PieceList, FormationType, SurroundMode, TroopPath, _facing_toward
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay #, SquadVisionOverlay
from HexMagic.game.piece import FormationType, _formation_offsets


from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay

from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds




from enum import Enum

In [ ]:
#| export
from HexMagic.game.food import FoodOverlay, FoodYield, PieceOverlay, PieceDataOverlay, PieceArrowOverlay, SquadSymbolOverlay

In [ ]:
#| export
import pyomo.environ as pyo
import optuna

Based on your integration notes, you need:

```bash
!pip install pyomo highspy
```

- **`pyomo`** — the modeling library itself
- **`highspy`** — the Python bindings for HiGHS, which lets Pyomo use it via `SolverFactory('appsi_highs')`

You already have `scipy` (which bundles HiGHS internally), but Pyomo talks to HiGHS through `highspy` directly rather than through scipy's wrapper.

After installing, verify with a quick smoke test:

```python
import pyomo.environ as pyo

m = pyo.ConcreteModel()
m.x = pyo.Var(bounds=(0, 10))
m.obj = pyo.Objective(expr=m.x, sense=pyo.maximize)

solver = pyo.SolverFactory('appsi_highs')
result = solver.solve(m)
print(f"Status: {result.solver.termination_condition}, x = {pyo.value(m.x)}")
```

If that prints `Status: optimal, x = 10.0` you're good to go.

In [ ]:
#| export
STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

In [ ]:
showDemo = False
myStuff = GameParts()

In [ ]:

TerrainDisplay(
    CreamOverlay(stylized=True),
    SquadSymbolOverlay([myStuff.queenGuard]),
    #SettlementOverlay(scale=1),
    #KingdomNamesOverlay(),
    PieceOverlay([myStuff.queenGuard]),
    SquadVisionOverlay([myStuff.queenGuard]),
    terrain=myStuff.terr,
    region=myStuff.region(), padding=2,
    board=myStuff.board,
    debug = not showDemo
)



In [ ]:
# Make sure soil exists

soil = SoilSystem.from_plates(myStuff.terr, [])

fy = FoodYield(myStuff.terr, myStuff.basin)
fy.compute()
#show(fy)
myStuff.fy = fy
fy.summary()

In [ ]:
show(fy)

## FoodOverlay

In [ ]:

myStuff.clearTerr()
myStuff.terr.repalette()
TerrainDisplay(
    TerrainOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(max_width=6),
    terrain=myStuff.terr,
    basins=myStuff.basin,
    debug = not showDemo
)


In [ ]:
#!cat ../../HexMagic/game/piece.py

In [ ]:
#| export
STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

## TroopPath

In [ ]:
myStuff.clearTerr()
myStuff.terr.repalette()
myStuff.grid.adjustRadius(35)


In [ ]:
showDemo = False

In [ ]:
??TerrainOverlay

In [ ]:
#read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

## The simulation

In [ ]:
#| export
from typing import Callable

@dataclass
class TurnReport:
    """End-of-turn context passed to squad callbacks."""
    turn: int
    squad: Squad
    snapshot: dict
    food_audit: dict
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    countries: np.ndarray | None
    elevation_mult: float
    sim_df: pd.DataFrame

SquadCallback = Callable[[TurnReport], None]


In [ ]:
#| export
# Add to SimEventType
class SimEventType(Enum):
    MOVED     = "moved"
    BLOCKED   = "blocked"
    BUMPED    = "bumped"      # ← new
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"
    ATTACKED  = "attacked"
    COUNTERED = "countered"


_FREE = {Instruction.ROT_L, Instruction.ROT_R}
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND,
              Instruction.HARVEST, Instruction.GIVE, Instruction.SETTLE}

@patch
def _eat_phase(self: FoodSimulator, acted_ids: set = None):
    for piece in list(self.pieces):
        if piece.health <= 0:
            continue

        shortfall = max(0.0, piece.diet - piece.food)
        piece.food = max(0.0, piece.food - piece.diet)
        self._log(piece, SimEventType.ATE, amount=piece.diet)

        if shortfall > 0:
            damage = shortfall * STARVE_MULT
            piece.health -= damage
            self._log(piece, SimEventType.STARVED, amount=damage)

        if piece.health <= 0:
            piece.health = 0
            self._log(piece, SimEventType.DIED)


In [ ]:
#| export
@dataclass
class FoodSimulator:
    """Turn-based simulator with per-slot interleaved execution."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list
    countries: np.ndarray = None
    elevation_mult: float = 0.005

    squads: list = field(default_factory=list)
    callbacks: dict = field(default_factory=dict)  # squad.id → SquadCallback

    _rows: list = field(default_factory=list)
    turn: int = 0
    snapshots: list = field(default_factory=list)

    def __post_init__(self):
        self._squad_map = {s.id: s for s in self.squads}

    @property
    def alive(self):
        return [p for p in self.pieces if p.health > 0]

    @property
    def occupied(self):
        return {p.location for p in self.alive
                if p.location is not None and p.location >= 0}

    @property
    def df(self) -> pd.DataFrame:
        return pd.DataFrame(self._rows)

    def _sort_move(self, pieces):
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99), p.birth_year, -p.health, p.id))

    # ── actions (unchanged) ──
    def _do_harvest(self, piece, slot=None):
        loc = piece.location
        if loc is None or loc < 0: return
        tier = int(self.food_tiers[loc])
        gained = piece.harvest_strength * tier
        old = piece.food
        piece.food = min(piece.food + gained, piece.food_capacity)
        self._log(piece, SimEventType.HARVESTED, slot=slot, amount=piece.food - old)

    def _do_give(self, piece, slot=None):
        reserve = FOOD_RESERVE_TURNS * piece.diet
        surplus = max(0.0, piece.food - reserve)
        if surplus <= 0: return
        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult, facing_only=True))
        receivers = sorted(
            [p for p in self.alive if p is not piece
             and p.owner_id == piece.owner_id
             and p.food < p.food_capacity
             and p.location in visible],
            key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))
        for recv in receivers:
            if surplus <= 0: break
            amount = min(surplus, recv.food_capacity - recv.food)
            recv.food += amount; piece.food -= amount; surplus -= amount
            self._log(piece, SimEventType.GAVE, slot=slot,
                      dst_piece_id=recv.id, amount=amount)


    def _eat_phase(self, acted_ids: set = None):
        for piece in list(self.pieces):
            if piece.health <= 0: continue
            shortfall = max(0.0, piece.diet - piece.food)
            piece.food = max(0.0, piece.food - piece.diet)
            self._log(piece, SimEventType.ATE, amount=piece.diet)
            if shortfall > 0:
                damage = shortfall * STARVE_MULT
                piece.health -= damage
                self._log(piece, SimEventType.STARVED, amount=damage)
            if piece.health <= 0:
                piece.health = 0
                self._log(piece, SimEventType.DIED)


    def _fire_callbacks(self):
        if not self.callbacks: return
        snap = self.snapshots[-1] if self.snapshots else {}
        for squad_id, cb in self.callbacks.items():
            squad = self._squad_map.get(squad_id)
            if squad is None or not squad.alive: continue
            report = TurnReport(
                turn=self.turn, squad=squad, snapshot=snap,
                food_audit=squad.food_audit(self.food_tiers),
                grid=self.grid, elevations=self.elevations,
                food_tiers=self.food_tiers, countries=self.countries,
                elevation_mult=self.elevation_mult, sim_df=self.df,
            )
            cb(report)


    def run(self, num_turns=20):
        for _ in range(num_turns):
            if not self.alive: self._snapshot(); break
            self.step()
        return self

    def summary(self):
        a = self.alive
        df = self.df
        print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
        if a:
            print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
                f"min={min(p.health for p in a):.0f}")
            print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
                f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")

        if df.empty: return

        # Moves per piece
        moves = df[df['event'] == 'moved'].groupby('piece_name').size()

        # Deaths
        deaths = df[df['event'] == 'died'][['turn', 'piece_name']]

        if not moves.empty:
            print("  Movement:")
            for name, n in moves.items():
                print(f"    {name}: {n} hexes")

        if not deaths.empty:
            print("  Deaths:")
            for _, row in deaths.iterrows():
                print(f"    {row.piece_name} died on turn {row.turn}")


In [ ]:
#| export
@patch
def _snapshot(self: FoodSimulator):
    alive = self.alive
    self.snapshots.append({
        'turn': self.turn,
        'alive': len(alive), 'total': len(self.pieces),
        'avg_health': np.mean([p.health for p in alive]) if alive else 0,
        'min_health': min((p.health for p in alive), default=0),
        'avg_food': np.mean([p.food for p in alive]) if alive else 0,
        'avg_turns_left': np.mean([p.food / max(p.diet, 0.01) for p in alive]) if alive else 0,
        'piece_states': {
            p.id: {'location': p.location, 'food': p.food,
                    'food_capacity': p.food_capacity, 'health': p.health,
                    'facing': p.facing}
            for p in self.pieces},
    })

In [ ]:
#| export
@patch
def _log(self: FoodSimulator, piece, etype: SimEventType, *,
         slot=None, src_hex=None, dst_hex=None,
         dst_piece_id=None, amount=0.0):
    loc = piece.location if piece.location is not None else -1
    self._rows.append({
        'turn': self.turn, 'slot': slot, 'event': etype.value,
        'piece_id': piece.id, 'piece_name': piece.name,
        'piece_type': piece.piece_type.value,
        'facing': piece.facing,
        'src_hex': src_hex if src_hex is not None else loc,
        'dst_hex': dst_hex if dst_hex is not None else loc,
        'dst_piece_id': dst_piece_id, 'amount': amount,
    })


In [ ]:
#| export
@patch
def _try_bump(self: FoodSimulator, mover, blocker, occupied, slot=None):
    """Swap mover (higher rank) with blocker (lower rank).
    
    Prepends recovery instructions to blocker:
      rotate → face old hex, FORWARD, rotate → restore original facing.
    Returns movement cost for the mover, or None if bump isn't allowed.
    """
    # Must be allies
    if mover.owner_id != blocker.owner_id:
        return None
    # Strictly higher rank
    if RANK_ORDER.get(mover.piece_type, 99) <= RANK_ORDER.get(blocker.piece_type, 99):
        return None
    
    old_mover_loc = mover.location
    old_blocker_loc = blocker.location
    old_blocker_facing = blocker.facing
    
    # ── Swap positions ──
    occupied.discard(old_mover_loc)
    occupied.discard(old_blocker_loc)
    mover.location = old_blocker_loc
    blocker.location = old_mover_loc
    occupied.add(old_blocker_loc)
    occupied.add(old_mover_loc)
    
    cost = _move_cost(self.elevations, old_mover_loc, old_blocker_loc)
    
    # ── Build recovery instructions for the bumped piece ──
    # Step 1: rotate to face old position (where it wants to return)
    target_facing = _facing_toward(self.grid, old_mover_loc, old_blocker_loc)
    
    diff = (target_facing - old_blocker_facing) % 6
    if diff <= 3:
        rot_to = [Instruction.ROT_R.value] * diff
    else:
        rot_to = [Instruction.ROT_L.value] * (6 - diff)
    
    # Step 2: FORWARD (will likely be blocked this turn, succeeds next)
    
    # Step 3: rotate back to original facing
    diff_back = (old_blocker_facing - target_facing) % 6
    if diff_back <= 3:
        rot_back = [Instruction.ROT_R.value] * diff_back
    else:
        rot_back = [Instruction.ROT_L.value] * (6 - diff_back)
    
    recovery = rot_to + [Instruction.FORWARD.value] + rot_back
    
    # Insert at current cursor (so recovery executes before existing plan)
    il = blocker.instructions
    il.rules = il.rules[:il.cursor] + recovery + il.rules[il.cursor:]
    # cursor stays — recovery is now what's next
    
    # ── Log ──
    self._log(mover, SimEventType.BUMPED, slot=slot,
              src_hex=old_mover_loc, dst_hex=old_blocker_loc,
              dst_piece_id=blocker.id)
    
    return cost


In [ ]:
#| export
@patch
def _try_forward(self: FoodSimulator, piece, occupied, slot=None, budget=float('inf')):
    d = HexPosition.directions()[piece.facing % 6]
    nbr = self.grid.hexposition_to_index(d, piece.location)
    
    passable = (0 <= nbr < len(self.elevations)
                and nbr not in self.grid.invalidRegion
                and self.elevations[nbr] >= 1)
    
    if not passable:
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    if nbr in occupied and nbr != piece.location:
        blocker = next((p for p in self.alive if p.location == nbr), None)
        if blocker:
            bump_cost = self._try_bump(piece, blocker, occupied, slot)
            if bump_cost is not None:
                return bump_cost
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    cost = _move_cost(self.elevations, piece.location, nbr)
    if cost > budget:
        return 0.0  # Can't afford — don't move!
    
    src = piece.location
    occupied.discard(src); piece.location = nbr; occupied.add(nbr)
    self._log(piece, SimEventType.MOVED, slot=slot, src_hex=src, dst_hex=nbr, amount=cost)
    return cost

In [ ]:
#| export
@patch
def _tick_one(self: FoodSimulator, piece, budget, occupied, slot=None):
    il = piece.instructions
    if not il.rules: return None, 0.0
    while True:
        if il.cursor >= len(il.rules):
            if il.patrol: il.cursor = 0
            else: return None, 0.0
        instr = Instruction(il.rules[il.cursor])
        if instr in _FREE:
            piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                            else (piece.facing + 1)) % 6
            self._log(piece, SimEventType.ROTATED, slot=slot)
            il.cursor += 1; continue
        if instr in _COSTS_ONE:
            if budget < 1.0: return None, 0.0
            il.cursor += 1
            if instr == Instruction.HARVEST: self._do_harvest(piece, slot)
            elif instr == Instruction.GIVE: self._do_give(piece, slot)
            return instr, 1.0
        # FORWARD — pass budget so _try_forward won't move if we can't afford it
        if budget < 1.0: return None, 0.0
        cost = self._try_forward(piece, occupied, slot, budget=budget)
        if cost > 0:
            il.cursor += 1
            return Instruction.FORWARD, cost
        return None, 0.0

In [ ]:
#| export
@patch
def _combat_phase(self: FoodSimulator, moved_ids: set, defended_ids: set):
    """Stationary pieces roll temperament; attackers flame all enemies in sight cone."""
    for piece in list(self.alive):
        if piece.health <= 0:
            continue
        if piece.id in moved_ids:
            continue

        if random.randint(0, 99) < getattr(piece, 'temperment', 100):
            continue

        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult, facing_only=True))

        enemies = [p for p in self.alive
                   if p.owner_id != piece.owner_id
                   and p.health > 0
                   and p.location in visible]

        if not enemies:
            continue

        for target in enemies:
            damage = piece.attack_strength / max(target.defense, 0.01)
            target.health = max(0, target.health - damage)
            self._log(piece, SimEventType.ATTACKED,
                      dst_piece_id=target.id, amount=damage)

            # Counter-attack: only if defending AND can see attacker
            if target.id in defended_ids and target.health > 0:
                target_vis = set(target.hexes_in_sight(
                    self.grid, self.elevations,
                    elevation_mult=self.elevation_mult, facing_only=True))
                if piece.location in target_vis:
                    counter = target.attack_strength / max(piece.defense, 0.01)
                    piece.health = max(0, piece.health - counter)
                    self._log(target, SimEventType.COUNTERED,
                              dst_piece_id=piece.id, amount=counter)

            if target.health <= 0:
                self._log(target, SimEventType.DIED)
            if piece.health <= 0:
                self._log(piece, SimEventType.DIED)
                break


In [ ]:
#| export


# ── Updated step() — tracks moved/defended, calls combat after eat ──
@patch
def step(self: FoodSimulator):
    alive = self.alive
    if not alive: return
    max_slots = max(p.move_strength for p in alive)
    budgets = {p.id: float(p.move_strength) for p in alive}
    acted = set()
    moved_ids = set()
    defended_ids = set()
    occupied = self.occupied
    order = self._sort_move(alive)

    for slot in range(max_slots):
        for piece in order:
            if piece.health <= 0: continue
            if budgets[piece.id] <= 0: continue
            instr, cost = self._tick_one(piece, budgets[piece.id], occupied, slot=slot)
            if instr is not None:
                budgets[piece.id] -= cost
                acted.add(piece.id)
                if instr == Instruction.FORWARD:
                    moved_ids.add(piece.id)
                elif instr == Instruction.DEFEND:
                    defended_ids.add(piece.id)

    self._eat_phase()
    self._combat_phase(moved_ids, defended_ids)
    self._snapshot()
    self._fire_callbacks()
    self.turn += 1


I need to add facing into the event logs can you provide patches

```markdown
Yes, much more. A few things jump out now:

Already there (the "actual" schedule):

    sim.df has per-turn event logs — that's the raw actual timeline
    sim.snapshots has per-piece state (location, food, health) every turn — that's the state view

Missing for our Schedule schema:

    Facing — the simulator updates it but doesn't store it in snapshots or the log
    Corridor/segment context — the simulator has no idea which corridor a piece is serving. It just runs instructions blindly.

The design implication: The simulator doesn't need to change much — we just need a thin adapter that reads sim.snapshots + sim.df and writes rows into our Schedule DataFrame format. The corridor/segment mapping can be done after the fact by looking up each piece's location against known corridors.

The facing gap is the only thing that probably needs a small change inside _snapshot() — adding 'facing': piece.facing to the piece_states dict.

Does that match your read of it?
``

### Callbacks

In [ ]:
#| export
def food_triage(report: TurnReport):
    """Prepend harvest+give when squad is hungry."""
    avg_left = report.snapshot.get('avg_turns_left', 999)
    if avg_left >= FOOD_RESERVE_TURNS:
        return
    
    alive = [p for p in report.squad.pieces if p.health > 0]
    for piece in alive:
        loc = piece.location
        if loc is None or loc < 0:
            continue
        
        prepend = []
        tier = int(report.food_tiers[loc]) if 0 <= loc < len(report.food_tiers) else 0
        if tier >= 1 and piece.food < piece.food_capacity:
            prepend.append(Instruction.HARVEST.value)
        
        reserve = FOOD_RESERVE_TURNS * piece.diet
        if piece.food > reserve:
            visible = piece.hexes_in_sight(
                report.grid, report.elevations,
                elevation_mult=report.elevation_mult,
                facing_only=True)
            hungry = any(
                p for p in alive
                if p is not piece and p.location in visible
                and p.food < p.food_capacity
            )
            if hungry:
                prepend.append(Instruction.GIVE.value)
        
        if prepend:
            piece.instructions.rules = prepend + piece.instructions.rules


In [ ]:
def compose_callbacks(*fns: SquadCallback) -> SquadCallback:
    """Chain multiple callbacks — each sees mutations from the previous."""
    def combined(report: TurnReport):
        for fn in fns:
            fn(report)
    return combined


### Analysis

#| export
@patch
def __ft__(self: FoodSimulator):
    """Line charts: alive count, health, food turns remaining."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    df = pd.DataFrame(self.snapshots)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))

    # Alive
    axes[0].plot(df['turn'], df['alive'], 'o-', color='#2ecc71', lw=2, ms=4)
    axes[0].axhline(df['total'].iloc[0], color='#bbb', ls='--', lw=1)
    axes[0].set_title('Alive', fontweight='bold', fontsize=10)
    axes[0].set_xlabel('Turn'); axes[0].set_ylim(bottom=0)
    axes[0].fill_between(df['turn'], df['alive'], alpha=0.15, color='#2ecc71')

    # Health
    axes[1].plot(df['turn'], df['avg_health'], 'o-', color='#e74c3c', lw=2, ms=4, label='avg')
    axes[1].plot(df['turn'], df['min_health'], 's--', color='#c0392b', lw=1, ms=3, alpha=0.6, label='min')
    axes[1].set_title('Health', fontweight='bold', fontsize=10)
    axes[1].set_xlabel('Turn'); axes[1].legend(fontsize=8); axes[1].set_ylim(bottom=0)

    # Food turns remaining
    axes[2].plot(df['turn'], df['avg_turns_left'], 'o-', color='#f39c12', lw=2, ms=4)
    axes[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1, label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes[2].set_title('Food (turns left)', fontweight='bold', fontsize=10)
    axes[2].set_xlabel('Turn'); axes[2].legend(fontsize=8); axes[2].set_ylim(bottom=0)

    for ax in axes:
        ax.grid(axis='y', alpha=0.3, lw=0.5)
        ax.tick_params(labelsize=8)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")

In [ ]:
#| export
@patch
def __ft__(self: FoodSimulator):
    """Line charts by piece type + activity breakdown over time."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    pinfo = {p.id: (p.piece_type, p.diet) for p in self.pieces}
    all_ranks = sorted({pt for pt, _ in pinfo.values()},
                       key=lambda p: RANK_ORDER.get(p, 99))

    rank_colors = {
        PieceType.KING:   '#9b59b6', PieceType.QUEEN:  '#e74c3c',
        PieceType.ROOK:   '#3498db', PieceType.BISHOP: '#2ecc71',
        PieceType.KNIGHT: '#f39c12', PieceType.PAWN:   '#95a5a6',
    }

    turns = [s['turn'] for s in self.snapshots]
    rank_data = {pt: {'alive': [], 'avg_health': [], 'avg_turns_left': []}
                 for pt in all_ranks}

    for snap in self.snapshots:
        by_rank = {pt: {'alive': 0, 'healths': [], 'turns_left': []}
                   for pt in all_ranks}
        for pid, state in snap['piece_states'].items():
            pt, diet = pinfo.get(pid, (None, 1.0))
            if pt is None: continue
            if state['health'] > 0:
                by_rank[pt]['alive'] += 1
                by_rank[pt]['healths'].append(state['health'])
                by_rank[pt]['turns_left'].append(state['food'] / max(diet, 0.01))
        for pt in all_ranks:
            d = by_rank[pt]
            rank_data[pt]['alive'].append(d['alive'])
            rank_data[pt]['avg_health'].append(
                np.mean(d['healths']) if d['healths'] else 0)
            rank_data[pt]['avg_turns_left'].append(
                np.mean(d['turns_left']) if d['turns_left'] else 0)

    # ── Activity data — tactical events only (skip routine move/rotate/eat) ──
    tactical_events = [
        SimEventType.HARVESTED, SimEventType.GAVE,
        SimEventType.ATTACKED,  SimEventType.COUNTERED,
        SimEventType.BUMPED,    SimEventType.STARVED, SimEventType.DIED,
    ]
    event_colors = {
        SimEventType.HARVESTED: '#2ecc71', SimEventType.GAVE:      '#1abc9c',
        SimEventType.ATTACKED:  '#e74c3c', SimEventType.COUNTERED: '#c0392b',
        SimEventType.BUMPED:    '#9b59b6', SimEventType.STARVED:   '#f39c12',
        SimEventType.DIED:      '#2c3e50',
    }
    max_turn = max(turns) if turns else 0
    event_counts = {et: np.zeros(max_turn + 1) for et in tactical_events}
    ev_lookup = {et.value: et for et in tactical_events}
    for row in self._rows:
        et = ev_lookup.get(row['event'])
        if et and row['turn'] <= max_turn:
            event_counts[et][row['turn']] += 1

    active = [et for et in tactical_events if event_counts[et].any()]

    # ── Layout: 3 rank charts on top, stacked bar below ──
    fig = plt.figure(figsize=(11, 6.5))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.85], hspace=0.4)
    axes_top = [fig.add_subplot(gs[0, i]) for i in range(3)]
    ax_act = fig.add_subplot(gs[1, :])

    # ── Top row: rank lines ──
    for pt in all_ranks:
        c = rank_colors.get(pt, '#333'); lbl = pt.name.title()
        d = rank_data[pt]
        axes_top[0].plot(turns, d['alive'], 'o-', color=c, lw=2, ms=3, label=lbl)
        axes_top[1].plot(turns, d['avg_health'], 'o-', color=c, lw=2, ms=3, label=lbl)
        axes_top[2].plot(turns, d['avg_turns_left'], 'o-', color=c, lw=2, ms=3, label=lbl)

    axes_top[0].set_title('Alive by Rank', fontweight='bold', fontsize=10)
    axes_top[0].set_xlabel('Turn'); axes_top[0].set_ylim(bottom=0)
    axes_top[1].set_title('Avg Health by Rank', fontweight='bold', fontsize=10)
    axes_top[1].set_xlabel('Turn'); axes_top[1].set_ylim(bottom=0)
    axes_top[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1,
                        label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes_top[2].set_title('Food (turns left) by Rank', fontweight='bold', fontsize=10)
    axes_top[2].set_xlabel('Turn'); axes_top[2].set_ylim(bottom=0)
    for ax in axes_top:
        ax.legend(fontsize=7, loc='best')
        ax.grid(axis='y', alpha=0.3, lw=0.5); ax.tick_params(labelsize=8)

    # ── Bottom row: stacked bar of tactical events ──
    turn_range = np.arange(max_turn + 1)
    if active:
        bottoms = np.zeros(max_turn + 1)
        for et in active:
            vals = event_counts[et]
            ax_act.bar(turn_range, vals, bottom=bottoms, width=0.8,
                       color=event_colors.get(et, '#999'),
                       label=et.value.title(), alpha=0.85)
            bottoms += vals

    ax_act.set_title('Tactical Actions per Turn', fontweight='bold', fontsize=10)
    ax_act.set_xlabel('Turn'); ax_act.set_ylabel('Count')
    ax_act.legend(fontsize=7, loc='upper right',
                  ncol=min(len(active), 4) if active else 1)
    ax_act.grid(axis='y', alpha=0.3, lw=0.5); ax_act.tick_params(labelsize=8)
    ax_act.set_ylim(bottom=0)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")


can you rewrite so that FoodSimulator.__ft__ so that it does it by rank (so there would be a line for all of the pawns, a line for all of the queens, etc)?

and maybe we have a chart below on the _ft_ which shows the actions as dot sizes or maybebar based upon round SimEventType. This categorical data that we want to see how it changes over time. so I am happy if you have a better visualiztion. It can be the case that per round a piece could do mulitple actions and it is okay to include it in both, we just want to see how much attacking, giving are doing over time.

did I put these in correctly?

In [ ]:
#| export
@patch
def squad_chart(self: FoodSimulator, squad: Squad = None, figsize=(10, 4)):
    """Per-piece line chart of health & food over turns."""
    if not self.snapshots:
        print("No snapshots yet"); return

    pieces = squad.pieces if squad else self.pieces
    pid_set = {p.id for p in pieces}
    pid_names = {p.id: f"{p.name} ({p.piece_type.value})" for p in pieces}

    turns = [s['turn'] for s in self.snapshots]

    # Extract per-piece time series from existing snapshots
    health = {pid: [] for pid in pid_set}
    food   = {pid: [] for pid in pid_set}

    for snap in self.snapshots:
        states = snap.get('piece_states', {})
        for pid in pid_set:
            st = states.get(pid)
            health[pid].append(st['health'] if st else 0)
            food[pid].append(st['food']     if st else 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    for pid in pid_set:
        lbl = pid_names.get(pid, pid[:8])
        ax1.plot(turns, health[pid], 'o-', ms=3, lw=1.5, label=lbl)
        ax2.plot(turns, food[pid],   'o-', ms=3, lw=1.5, label=lbl)

    ax1.set_title('Health per turn', fontweight='bold', fontsize=10)
    ax1.set_xlabel('Turn'); ax1.set_ylim(bottom=0)
    ax1.legend(fontsize=7, loc='lower left')
    ax1.grid(axis='y', alpha=0.3)

    ax2.set_title('Food per turn', fontweight='bold', fontsize=10)
    ax2.set_xlabel('Turn'); ax2.set_ylim(bottom=0)
    ax2.legend(fontsize=7, loc='upper right')
    ax2.grid(axis='y', alpha=0.3)

    fig.tight_layout()
    plt.show()


In [ ]:
#| export
@patch
def overlayContext(parts: GameParts, *, region=None, padding=1, radius=None,
                   corridors=None, extra_squads=None, extra_pieces=None,
                   simulator=None):
    """Build a GameContext from a GameParts instance."""
    terrain = parts.terr
    if region is not None:
        terrain = terrain.zoom(region, padding=padding)

    grid = terrain.hexGrid
    if radius:
        grid.adjustRadius(radius)

    squads = list(getattr(parts, 'squads', []))
    if extra_squads:
        squads.extend(extra_squads)

    pieces = []
    for sq in squads:
        pieces.extend(sq.alive if hasattr(sq, 'alive') else sq.pieces)
    if extra_pieces:
        pieces.extend(extra_pieces)

    return GameContext(
        terrain=terrain, grid=grid, builder=grid.builder,
        extras=dict(
            board=parts.board,
            basins=parts.basin,
            corridors=corridors or [],
            pieces=pieces,
            squads=squads,
            simulator=simulator,
            c2f=getattr(terrain, 'c2f', None),
        )
    )


In [ ]:
??SquadVisionOverlay

In [ ]:
#| export
def SquadFoodTrailOverlay(min_opacity: float = 0.06,
                          max_opacity: float = 0.65,
                          **kw) -> OverlaySpec:
    """Temporal food trail — pip circles sized by food %, arrows for shares, X for deaths.

    Opacity fades forward: earliest turns are faintest, most recent are boldest.
    Uses DiagramGlyphs for consistent styling with the rest of the overlay system.

    Reads simulator from ctx.simulator and squads from ctx.squads (GameContext).
    Shows all squad pieces found in context.
    """

    def render(ctx) -> str:
        sim = ctx.simulator
        if sim is None:
            return ''

        grid    = ctx.grid
        builder = ctx.builder
        c2f     = ctx.extras.get('c2f')

        board = ctx.board
        if board is not None:
            sim_grid = board.terrain.hexGrid
        else:
            sim_grid = grid

        # ── helper: coarse index → list of fine indices ──
        def _c2f_list(idx):
            if c2f is None:
                return [idx]
            val = c2f.get(idx)
            if val is None:
                return []
            return val if isinstance(val, list) else [val]

        def _c2f_first(idx):
            mapped = _c2f_list(idx)
            return mapped[0] if mapped else -1

        # ── Collect pieces from ctx.squads ──
        squads = ctx.squads
        if squads:
            src_pieces = []
            for sq in squads:
                src_pieces.extend(sq.pieces if not hasattr(sq, 'alive') else sq.pieces)
            piece_ids = {p.id for p in src_pieces}
        else:
            src_pieces = sim.pieces
            piece_ids = {p.id for p in src_pieces}

        snaps = sim.snapshots
        if not snaps:
            return ''
        max_turn = max(s['turn'] for s in snaps) or 1

        # Event log
        df = sim.df
        gives  = df[(df.event == 'gave')  & (df.piece_id.isin(piece_ids))]
        deaths = df[(df.event == 'died')  & (df.piece_id.isin(piece_ids))]

        r = grid.radius if hasattr(grid, 'radius') else 20
        N = len(grid.hexes)

        # Build one DiagramGlyphs per flag we encounter, register styles once
        flag_glyphs: dict = {}
        def _get_glyphs(flag):
            key = id(flag) if flag else 'default'
            if key not in flag_glyphs:
                g = DiagramGlyphs(flag or CountryFlag.seaborn("husl", 1)[0], size=r * 0.7)
                g.register_styles(builder)
                for pct in range(0, 101, 5):
                    f = pct / 100
                    color = StyleCSS.lerp_color('#e74c3c', '#27ae60', f)
                    builder.add_style(StyleCSS(
                        f"pip_{pct}", fill=color, stroke=color,
                        stroke_width=0.8, opacity=0.85))
                flag_glyphs[key] = g
            return flag_glyphs[key]

        # Map piece_id → flag for glyph lookup
        piece_flags = {p.id: p.flag for p in src_pieces}

        def _render_idx(loc):
            if loc is None or loc < 0:
                return -1
            ri = _c2f_first(loc) if c2f else loc
            return ri if 0 <= ri < N else -1

        parts = []

        # ── Per-turn food pips + share arrows ──
        for snap in snaps:
            t = snap['turn']
            alpha = min_opacity + (max_opacity - min_opacity) * (t / max_turn)
            states = snap.get('piece_states', {})

            # Food pips
            for pid, st in states.items():
                if pid not in piece_ids or st['health'] <= 0:
                    continue
                ri = _render_idx(st['location'])
                if ri < 0:
                    continue

                c = grid.hexes[ri].center
                cap = max(st['food_capacity'], 0.01)
                frac = np.clip(st['food'] / cap, 0, 1)

                glyphs = _get_glyphs(piece_flags.get(pid))
                pip_svg = glyphs.food_pip(MapCord(c.x, c.y), float(frac))
                parts.append(f'<g opacity="{alpha:.2f}">{pip_svg}</g>')

            # Give arrows for this turn
            turn_gives = gives[gives.turn == t]
            for _, row in turn_gives.iterrows():
                g_st = states.get(row.piece_id)
                r_st = states.get(row.dst_piece_id)
                if not g_st or not r_st:
                    continue
                gi = _render_idx(g_st['location'])
                ri2 = _render_idx(r_st['location'])
                if gi < 0 or ri2 < 0 or gi == ri2:
                    continue

                gc = grid.hexes[gi].center
                rc = grid.hexes[ri2].center
                glyphs = _get_glyphs(piece_flags.get(row.piece_id))
                arrow_svg = glyphs.transfer_arrow(
                    MapCord(gc.x, gc.y), MapCord(rc.x, rc.y))
                parts.append(f'<g opacity="{alpha:.2f}">{arrow_svg}</g>')

        # ── Death X markers (always prominent) ──
        for _, row in deaths.iterrows():
            ri = _render_idx(int(row.src_hex))
            if ri < 0:
                continue
            c = grid.hexes[ri].center
            glyphs = _get_glyphs(piece_flags.get(row.piece_id))
            parts.append(glyphs.death_marker(MapCord(c.x, c.y)))

        return '\n'.join(parts)

    return OverlaySpec("food_trail", render, requires={'simulator', 'squads'}, priority=68)


We should pull more from the context for SquadFoodTrailOverlay like SquadVisionOverlay did

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call. `SquadFoodTrailOverlay` hardcodes `squad` as a closure param and uses a bare `_map_point` instead of the `_c2f_list`/`_c2f_first` pattern. Here's the updated version that pulls from `ctx` the same way `SquadVisionOverlay` does:

```python
def SquadFoodTrailOverlay(min_opacity: float = 0.06,
                          max_opacity: float = 0.65,
                          **kw) -> OverlaySpec:
    """Temporal food trail — pip circles sized by food %, arrows for shares, X for deaths.

    Opacity fades forward: earliest turns are faintest, most recent are boldest.
    Uses DiagramGlyphs for consistent styling with the rest of the overlay system.

    Reads simulator from ctx.simulator and squads from ctx.squads (GameContext).
    Shows all squad pieces found in context.
    """

    def render(ctx) -> str:
        sim = ctx.simulator
        if sim is None:
            return ''

        grid    = ctx.grid
        builder = ctx.builder
        c2f     = ctx.extras.get('c2f')

        board = ctx.board
        if board is not None:
            sim_grid = board.terrain.hexGrid
        else:
            sim_grid = grid

        # ── helper: coarse index → list of fine indices ──
        def _c2f_list(idx):
            if c2f is None:
                return [idx]
            val = c2f.get(idx)
            if val is None:
                return []
            return val if isinstance(val, list) else [val]

        def _c2f_first(idx):
            mapped = _c2f_list(idx)
            return mapped[0] if mapped else -1

        # ── Collect pieces from ctx.squads ──
        squads = ctx.squads
        if squads:
            src_pieces = []
            for sq in squads:
                src_pieces.extend(sq.pieces if not hasattr(sq, 'alive') else sq.pieces)
            piece_ids = {p.id for p in src_pieces}
        else:
            src_pieces = sim.pieces
            piece_ids = {p.id for p in src_pieces}

        snaps = sim.snapshots
        if not snaps:
            return ''
        max_turn = max(s['turn'] for s in snaps) or 1

        # Event log
        df = sim.df
        gives  = df[(df.event == 'gave')  & (df.piece_id.isin(piece_ids))]
        deaths = df[(df.event == 'died')  & (df.piece_id.isin(piece_ids))]

        r = grid.radius if hasattr(grid, 'radius') else 20
        N = len(grid.hexes)

        # Build one DiagramGlyphs per flag we encounter, register styles once
        flag_glyphs: dict = {}
        def _get_glyphs(flag):
            key = id(flag) if flag else 'default'
            if key not in flag_glyphs:
                g = DiagramGlyphs(flag or CountryFlag.seaborn("husl", 1)[0], size=r * 0.7)
                g.register_styles(builder)
                for pct in range(0, 101, 5):
                    f = pct / 100
                    color = StyleCSS.lerp_color('#e74c3c', '#27ae60', f)
                    builder.add_style(StyleCSS(
                        f"pip_{pct}", fill=color, stroke=color,
                        stroke_width=0.8, opacity=0.85))
                flag_glyphs[key] = g
            return flag_glyphs[key]

        # Map piece_id → flag for glyph lookup
        piece_flags = {p.id: p.flag for p in src_pieces}

        def _render_idx(loc):
            if loc is None or loc < 0:
                return -1
            ri = _c2f_first(loc) if c2f else loc
            return ri if 0 <= ri < N else -1

        parts = []

        # ── Per-turn food pips + share arrows ──
        for snap in snaps:
            t = snap['turn']
            alpha = min_opacity + (max_opacity - min_opacity) * (t / max_turn)
            states = snap.get('piece_states', {})

            # Food pips
            for pid, st in states.items():
                if pid not in piece_ids or st['health'] <= 0:
                    continue
                ri = _render_idx(st['location'])
                if ri < 0:
                    continue

                c = grid.hexes[ri].center
                cap = max(st['food_capacity'], 0.01)
                frac = np.clip(st['food'] / cap, 0, 1)

                glyphs = _get_glyphs(piece_flags.get(pid))
                pip_svg = glyphs.food_pip(MapCord(c.x, c.y), float(frac))
                parts.append(f'<g opacity="{alpha:.2f}">{pip_svg}</g>')

            # Give arrows for this turn
            turn_gives = gives[gives.turn == t]
            for _, row in turn_gives.iterrows():
                g_st = states.get(row.piece_id)
                r_st = states.get(row.dst_piece_id)
                if not g_st or not r_st:
                    continue
                gi = _render_idx(g_st['location'])
                ri2 = _render_idx(r_st['location'])
                if gi < 0 or ri2 < 0 or gi == ri2:
                    continue

                gc = grid.hexes[gi].center
                rc = grid.hexes[ri2].center
                glyphs = _get_glyphs(piece_flags.get(row.piece_id))
                arrow_svg = glyphs.transfer_arrow(
                    MapCord(gc.x, gc.y), MapCord(rc.x, rc.y))
                parts.append(f'<g opacity="{alpha:.2f}">{arrow_svg}</g>')

        # ── Death X markers (always prominent) ──
        for _, row in deaths.iterrows():
            ri = _render_idx(int(row.src_hex))
            if ri < 0:
                continue
            c = grid.hexes[ri].center
            glyphs = _get_glyphs(piece_flags.get(row.piece_id))
            parts.append(glyphs.death_marker(MapCord(c.x, c.y)))

        return '\n'.join(parts)

    return OverlaySpec("food_trail", render, requires={'simulator', 'squads'}, priority=68)
```

The three key changes:

1. **Removed `squad` parameter** — reads squads from `ctx.squads` just like `SquadVisionOverlay`, so the same overlay spec works when squads change between renders.

2. **Added `_c2f_list`/`_c2f_first` helpers** — replaces the bare `_map_point` call with the same pattern `SquadVisionOverlay` uses, which properly handles missing mappings (returning `[]` / `-1`) and list-valued entries.

3. **Added `board` handling** — pulls `sim_grid` from `ctx.board.terrain.hexGrid` when available (currently stored but not used in the trail overlay, but having it consistent means if you later need sim-space lookups — e.g. for hex distances in arrow routing — it's already there).

4. **Updated `requires`** — added `'squads'` to the overlay spec's requirements alongside `'simulator'`.

In [ ]:
# Reset & apply food profiles (now that FOOD_RESERVE_TURNS exists)
  

# Find the queen
queen = [p for p in myStuff.queenGuard.pieces if p.piece_type == PieceType.QUEEN][0]
queen.facing = 1  # West

# Queen marches west
myStuff.queen.instructions = InstructionList(
    [Instruction.FORWARD.value] * 10, cursor=0, patrol=False)

# Pawns: harvest + give patrol (stationary food stations)
for p in myStuff.queenGuard.pieces:
    if p.piece_type == PieceType.PAWN:
        facing_to_queen = _facing_toward(myStuff.grid, p.location, myStuff.queen.location)
        p.facing = facing_to_queen
        p.instructions = InstructionList(
            [Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.HARVEST.value, Instruction.GIVE.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value],
            cursor=0, patrol=True)

        #p.instructions = []
        for i in range(0, 30):
            p.instructions.rules.append(Instruction.FORWARD.value)

## Dummy Data 
Lightweight piece proxy for planning simulations. The real `Piece` has DB backing, owner references, etc. — PieceShadow is a pure dataclass that the simulator can mutate freely without side effects.

In [ ]:
#| export
@dataclass
class PieceShadow:
    "Lightweight piece proxy for planning simulations."
    id:               str
    name:             str
    piece_type:       PieceType
    location:         int
    facing:           int
    birth_year:       int
    food:             float
    food_capacity:    float
    diet:             float
    health:           float
    max_health:       float
    move_strength:    int
    harvest_strength: float
    sight:            int
    owner_id:         int = 0
    flag:             CountryFlag = None
    instructions:     InstructionList = field(default_factory=lambda: InstructionList([], 0, False))
    # stubbed combat — planner doesn't fight
    attack_strength:  float = 0.0
    defense:          float = 1.0
    temperment:       int = 100

    @classmethod
    def from_piece(cls, piece):
        return cls(**{f: getattr(piece, f) for f in [
            'id','name','piece_type','birth_year','location','facing',
            'food','food_capacity','diet','health','max_health',
            'move_strength','harvest_strength','sight','owner_id','flag']})


In [ ]:
#| export
@patch
def hexes_in_sight(self: PieceShadow, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> set[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return set()

    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)

    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = HexPosition.origin().field_of_view( facing_dir, effective_sight)
        return {
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        }
    else:
        return set(grid.indices_in_range(self.location, effective_sight))

#| export
@patch
def loadGeology(self: GameParts, geo: Geology):
    self.terr = geo.terrain
    self.grid = self.terr.hexGrid
    self.builder = self.grid.builder

    self.board = GameBoard(self.terr, 2)
    self.basin = geo.basins

    self.country = self.board.kingdoms[0]
    self.home = self.country.settlements[0]
    self.squads = Squad.squads(6, self.country.flag)
    self.queenGuard = self.squads[0]
    for pType in [PieceType.PAWN, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN,
                  PieceType.PAWN, PieceType.BISHOP, PieceType.BISHOP, PieceType.QUEEN]:
        piece = self.home.recruit(pType, flag=self.queenGuard.flag)
        self.queenGuard.pieces.append(piece)

    # Initial placement so pieces have locations
    self.home.place_squad_spiral(self.queenGuard, self.terr)
    self.queen = self.queenGuard.by_rank()[0]

    # Set diet/harvest/capacity on all pieces
    self.apply_food_profile()

    # Compute food tiers, then re-place using sustainability logic
    fy = FoodYield(self.terr, self.basin)
    fy.compute()
    best_cost = smart_placement(self.home, self.queenGuard, self.terr, fy.tiers, self.grid)
    print(f"Smart placement: queen at {self.queen.location}, sustain cost = {best_cost} pawns")

## Placement - Optimal Camp Layout (Network-Flow MIP)

The hex neighborhood is modeled as a directed graph. Placement decides WHO goes WHERE, flow decides HOW MUCH food moves along each arc. Objective: maximize net surplus (total production - total diet), tiebreak by keeping high-combat pieces off food duty.

In [ ]:
#| export
def _solver_ok(result):
    return (result.solver.status == pyo.SolverStatus.ok and
            result.solver.termination_condition == pyo.TerminationCondition.optimal)

In [ ]:
#| export
def optimal_camp_layout(squad, center, grid, elevations, food_tiers,
                        countries=None, max_ring=3, elevation_mult=0.005):
    """Network-flow camp placement MIP.
    
    The hex neighborhood is a directed graph:
      - Nodes: candidate hexes (occupied by pieces)
      - Arcs: adjacency edges (or sight-cone edges for longer range)
      - Source nodes: pawns on food hexes (production = harvest × tier)
      - Relay nodes: bishops (high throughput, buffer capacity)
      - Sink nodes: queen/king (high diet, low harvest)
    
    Flow conservation at each occupied hex:
      production + inflow ≥ consumption + outflow
    
    Placement decides WHO goes WHERE.
    Flow decides HOW MUCH food moves along each arc.
    Objective: maximize net surplus (total prod - total diet),
               tiebreak by keeping high-combat pieces off food duty.
    """
    alive = squad.alive
    N = len(alive)
    
    # ── Candidate hexes (ring 0..max_ring around center) ──
    cands = []
    seen = set()
    for ring in range(0, max_ring + 1):
        ring_hps = [HexPosition.origin()] if ring == 0 else HexPosition.origin().ring(ring)
        for hp in ring_hps:
            idx = grid.hexposition_to_index(hp, center)
            if idx < 0 or idx in grid.invalidRegion or idx in seen: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            seen.add(idx)
            cands.append(idx)
    
    H = len(cands)
    tier = [int(food_tiers[c]) if 0 <= c < len(food_tiers) else 0
            for c in cands]
    hex_to_local = {c: i for i, c in enumerate(cands)}
    
    # ── Directed arcs: adjacency-based ──
    # (For sight-cone arcs, expand this with FOV precomputation)
    arcs = []
    for i in range(H):
        for nb in grid.neighborsOf(cands[i]):
            if nb in hex_to_local:
                j = hex_to_local[nb]
                arcs.append((i, j))
    
    # Piece parameters
    diet_p = [p.diet for p in alive]
    harv_p = [p.harvest_strength for p in alive]
    combat_p = [getattr(p, 'attack_strength', 0) for p in alive]
    
    # ── Pyomo model ──
    m = pyo.ConcreteModel()
    m.P = pyo.RangeSet(0, N - 1)
    m.H = pyo.RangeSet(0, H - 1)
    m.A = pyo.Set(initialize=arcs)
    
    # ── Decision variables ──
    m.place = pyo.Var(m.P, m.H, domain=pyo.Binary)    # piece p on hex h
    m.flow  = pyo.Var(m.A, domain=pyo.NonNegativeReals) # food along arc
    
    # ── Placement constraints ──
    m.one_hex = pyo.Constraint(m.P, rule=lambda m, p:
        sum(m.place[p, h] for h in m.H) == 1)
    
    m.one_piece = pyo.Constraint(m.H, rule=lambda m, h:
        sum(m.place[p, h] for p in m.P) <= 1)
    
    # ── Production / consumption at each hex ──
    def _prod(m, h):
        """Harvest output: Σ_p place[p,h] × harv_p × tier[h]"""
        return sum(m.place[p, h] * harv_p[p] * tier[h] for p in m.P)
    
    def _cons(m, h):
        """Diet demand: Σ_p place[p,h] × diet_p"""
        return sum(m.place[p, h] * diet_p[p] for p in m.P)
    
    # ── Flow conservation at every hex ──
    # production + inflow ≥ consumption + outflow
    def balance(m, h):
        inflow  = sum(m.flow[j, h] for (j, hh) in arcs if hh == h)
        outflow = sum(m.flow[h, j] for (hh, j) in arcs if hh == h)
        return _prod(m, h) + inflow >= _cons(m, h) + outflow
    m.balance = pyo.Constraint(m.H, rule=balance)
    
    # ── Arc capacity: can only send from an occupied hex ──
    M_big = max(harv_p) * max(tier + [1]) * 2
    m.arc_send = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, i] for p in m.P))
    m.arc_recv = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, j] for p in m.P))
    
    # ── Objective ──
    # Primary: maximize total surplus (production - consumption)
    # Secondary: minimize combat value on food hexes (keep fighters free)
    α = 100  # surplus is much more important than combat tiebreak
    
    # A piece is "on food duty" if on a hex with tier > 0
    food_duty_combat = sum(
        combat_p[p] * m.place[p, h] * (1 if tier[h] > 0 else 0)
        for p in range(N) for h in range(H))
    
    m.obj = pyo.Objective(
        expr=α * (sum(_prod(m, h) for h in m.H) 
                - sum(_cons(m, h) for h in m.H))
             - food_duty_combat,
        sense=pyo.maximize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, tee=False)
    if not _solver_ok(result):
        return None
    
    # ── Extract results ──
    placements = {}  # piece → hex_idx
    for p in m.P:
        for h in m.H:
            if pyo.value(m.place[p, h]) > 0.5:
                placements[alive[p].id] = cands[h]
                break
    
    flows = {}  # (hex_i, hex_j) → flow amount
    for (i, j) in arcs:
        v = pyo.value(m.flow[i, j])
        if v > 0.01:
            flows[(cands[i], cands[j])] = round(v, 2)
    
    total_prod = sum(pyo.value(_prod(m, h)) for h in m.H)
    total_cons = sum(pyo.value(_cons(m, h)) for h in m.H)
    
    return {
        'placements': placements,
        'flows': flows,
        'production': total_prod,
        'consumption': total_cons,
        'surplus': total_prod - total_cons,
        'cands': cands,
        'tiers': tier,
    }

### Smart Placement (two-phase)

Phase 1 scores candidate centers cheaply (sum of nearby food tiers). Phase 2 runs the full MIP on the top-k candidates. Phase 3 applies the best layout and faces pieces toward center.

In [ ]:
#| export
def smart_placement(settlement, squad, terrain, food_tiers, grid,
                       max_ring=10, camp_ring=3, top_k=5):
    """Place entire squad using optimal_camp_layout network-flow MIP.
    
    Phase 1: Score candidate centers cheaply (sum of nearby food tiers)
    Phase 2: Run full MIP on top-k candidates
    Phase 3: Apply best layout
    
    Returns the best result dict from optimal_camp_layout, or None.
    """
    center = settlement.location
    elevs = terrain.elevations
    
    # ── Phase 1: Cheap scoring of candidate centers ──
    candidates = []
    for hp in HexPosition.origin().spiral(max_ring):
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0 or elevs[idx] <= 0: continue
        
        # Quick heuristic: sum food tiers in the camp neighborhood
        score = 0
        land_count = 0
        for r in range(0, camp_ring + 1):
            ring_hps = [HexPosition.origin()] if r == 0 else HexPosition.origin().ring(r)
            for nhp in ring_hps:
                nidx = grid.hexposition_to_index(nhp, origin_index=idx)
                if nidx < 0 or elevs[nidx] <= 0: continue
                land_count += 1
                score += int(food_tiers[nidx]) if 0 <= nidx < len(food_tiers) else 0
        
        # Need enough land hexes for the squad
        if land_count >= len(squad.alive):
            candidates.append((idx, score))
    
    if not candidates:
        return None
    
    # Top-k by cheap score
    candidates.sort(key=lambda x: -x[1])
    shortlist = candidates[:top_k]
    
    # ── Phase 2: Full MIP on shortlist ──
    best_result = None
    best_surplus = -float('inf')
    best_center = center
    
    for cand_idx, cheap_score in shortlist:
        result = optimal_camp_layout(
            squad, cand_idx, grid, elevs, food_tiers,
            max_ring=camp_ring)
        
        if result is None: continue
        if result['surplus'] > best_surplus:
            best_surplus = result['surplus']
            best_result = result
            best_center = cand_idx
    
    if best_result is None:
        return None
    
    # ── Phase 3: Apply placements + facings ──
    for piece in squad.alive:
        hex_idx = best_result['placements'].get(piece.id)
        if hex_idx is not None:
            piece.location = hex_idx
            # Face toward camp center (for GIVE arcs)
            if hex_idx != best_center:
                piece.facing = _facing_toward(grid, hex_idx, best_center)
    
    best_result['center'] = best_center
    print(f"Smart placement v2: center={best_center}, "
          f"surplus={best_surplus:.1f} "
          f"(prod={best_result['production']:.1f}, "
          f"cons={best_result['consumption']:.1f}), "
          f"checked {len(shortlist)} candidates")
    
    return best_result

In [ ]:
#| export
from HexMagic.geology import Geology, DrainageBasins, SoilSystem
from HexMagic.climate import ClimatePreset, Climate, TerraDemo, TerrainFactory

In [ ]:
#| export
FOOD_PROFILE_V2 = {
    'pawn_diet':       0.7,
    'pawn_harvest':    1.3,
    'pawn_capacity':  14.0,
    'pawn_sight':      4,
    'queen_diet':      7.0,
    'queen_harvest':   0.2,
    'queen_capacity': 24.0,
    'bishop_diet':     3.5,
    'bishop_harvest':  0.7,
    'bishop_capacity': 140.0,
}

def apply_food_profile(pieces, profile=None):
    """Apply food profile to a list of pieces."""
    p = profile or FOOD_PROFILE_V2
    for piece in pieces:
        t = piece.piece_type.name.lower()
        for attr, field in [('diet','diet'), ('harvest','harvest_strength'),
                            ('capacity','food_capacity'), ('sight','sight')]:
            key = f'{t}_{attr}'
            if key in p:
                setattr(piece, field, p[key])
        # Fill to capacity
        piece.food = piece.food_capacity

@patch
def apply_food_profile(self: GameParts, soil_scale: float = 1.0):
    """Set food stats on all pieces based on balanced profile.
    
    Design intent:
      - Queen is expensive to feed (3.5× pawn), can't self-sustain
      - Pawns are field workers: decent harvest, small pack
      - Bishops are supply wagons: bad harvest, huge capacity
      - Knights are fast but hungry scouts
      - Rooks are heavy but carry well
      - King eats like a queen (royalty!)
    """
    PROFILES = {
        #                  diet  harvest  capacity
        PieceType.PAWN:   (2.0,  2.0,     12),
        PieceType.BISHOP: (2.5,  1.0,    100),
        PieceType.KNIGHT: (3.0,  1.5,     20),
        PieceType.ROOK:   (3.5,  1.0,     40),
        PieceType.QUEEN:  (7.0,  0.5,     30),
        PieceType.KING:   (6.0,  0.5,     25),
    }
    
    count = {t.name: 0 for t in PieceType}
    pieces = []
    for country in self.board.kingdoms:
        for city in country.settlements:
            for piece in city.citizens:
                profile = PROFILES.get(piece.piece_type)
                if profile is None:
                    continue
                piece.diet, piece.harvest_strength, piece.food_capacity = profile
                piece.food = piece.food_capacity  # start full
                count[piece.piece_type.name] += 1
                pieces.append(piece)

    apply_food_profile(pieces,FOOD_PROFILE_V2)
    
    # Apply soil scaling if needed

    fy = FoodYield(self.terr, self.basin)
    if soil_scale != 1.0:
        fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()

    
    print(f"Food profile applied:")
    for ptype, (d, h, c) in PROFILES.items():
        n = count.get(ptype.name, 0)
        net_t5 = 0.5 * h * 5 - d  # net at tier 5
        label = "✅" if net_t5 > 0 else "❌"
        print(f"  {ptype.name:8s} ×{n:2d}: diet={d:.1f} harvest={h:.1f} "
              f"cap={c:3d}  net@tier5={net_t5:+.1f} {label}")
    
    queen_d = PROFILES[PieceType.QUEEN][0]
    pawn_net = 0.5 * PROFILES[PieceType.PAWN][1] * 5 - PROFILES[PieceType.PAWN][0]
    bish_gap = PROFILES[PieceType.BISHOP][2] / (PROFILES[PieceType.BISHOP][0] + queen_d)
    print(f"\n  Pawns to sustain queen @tier5: {math.ceil(queen_d / max(pawn_net, 0.01))}")
    print(f"  Bishop gap crossing: ~{int(bish_gap)} barren hexes")
    return fy

In [ ]:
#| export
@patch
def loadGeology(self: GameParts, geo: Geology):
    self.terr = geo.terrain
    self.grid = self.terr.hexGrid
    self.builder = self.grid.builder

    self.board = GameBoard(self.terr, 2)
    self.basin = geo.basins

    self.country = self.board.kingdoms[0]
    self.home = self.country.settlements[0]
    self.squads = Squad.squads(6, self.country.flag)
    self.queenGuard = self.squads[0]
    for pType in [PieceType.PAWN, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN,
                  PieceType.PAWN, PieceType.BISHOP, PieceType.BISHOP, PieceType.QUEEN]:
        piece = self.home.recruit(pType, flag=self.queenGuard.flag)
        self.queenGuard.pieces.append(piece)

    # Initial placement so pieces have locations
    self.home.place_squad_spiral(self.queenGuard, self.terr)
    self.queen = self.queenGuard.by_rank()[0]

    # Set diet/harvest/capacity on all pieces
    self.apply_food_profile()

    # Compute food tiers, then re-place using sustainability logic
    fy = FoodYield(self.terr, self.basin)
    fy.compute()
    best_cost = smart_placement(self.home, self.queenGuard, self.terr, fy.tiers, self.grid)
    print(f"Smart placement: queen at {self.queen.location}, sustain cost = {best_cost} pawns")

In [ ]:
#| export
#sim.squad_chart(myStuff.queenGuard)
def knownWorld():

    world = TerrainFactory.create_world(
        bounds= MapRect(MapCord(0, 0), MapSize(300, 300)),
        preset='temperate',
        name='Maiden Lane',
        radius=15,
        lon_span=10.0,
        num_plates=8,
        subdivisions=3,
        ocean_fraction=0.3,
        oceanic_sides=['N'],  # Ocean on east and west
        terrain_age='young',  # Sharp, dramatic features
        formation_type='ridge',  # Creates ridge formations
        elevation_scale=1.5,  # Exaggerate the heights
        erosion_age=0.1,  # Minimal erosion for sharp peaks
        num_lakes=0,
        seed=23,
        debug=True
    )

    sampleMap = world.terrain
    sampleMap.repalette()

    for i in range(len(sampleMap.hexGrid.hexes)):
        sampleMap.hexGrid.hexes[i].label = str(i)

    return world

In [ ]:
myStuff = GameParts()
myStuff.loadGeology(knownWorld())
myStuff.grid.adjustRadius(40)

## Recording

I think we want the food simulator to generate a pandas dataframe of the actions.

Lets start to add combat to the food simulator. It should be after eat and move (always better to fight if you don't have an empty stomach)

I have added a temperement to piece which is how peacefull they are 100 = never attack 0 = always. If you haven't moved (ie you are paused, defend, settle) you do a saving throw for all opponents in your sight cone to see whether you attack. I do think we have some offense - defense kind of approach. If you have picked defend you get an attack back at them

```
dataclass
class Piece:
    """A game piece representing a group of units."""

    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0
    settlement_id: Optional[str] = None
    location: int = None
    name: str = ""
    birth_year: int = 1900

    # Core attributes
    size: int = 100
    health: int = 100
    max_health: int = 100

    # Vision
    sight: int = 3

    # Movement
    movement_range: int = 4
    current_position: int = -1

    # Facing & rules
    facing: int = 0
    instructions: InstructionList = field(default_factory=lambda: InstructionList([], 0, True))

    # Combat
    attack_strength: float = 1.0
    defense: float = 1.0
    move_strength: int = 3
    harvest_strength: float = 1.0
    temperment:int = 100

    # Type
    piece_type: PieceType = PieceType.PAWN
    squad: int = -1

    # Food
    food: float = 0.0
    diet: float = 1.0
    food_capacity: float = 10.0

    # Settlement state
    harvest_goal: Resources = None
    settle_progress: int = 0
    settle_threshold: int = 3

    # Non-serialised
    db: GeoStorage = None
    flag: CountryFlag = None

    PIECE_FIELDS = [
        'id', 'owner_id', 'settlement_id', 'location', 'name', 'birth_year',
        'size', 'health', 'max_health',
        'sight', 'movement_range', 'current_position',
        'facing', 'rules', 'cursor', 'patrol', 'squad',
        'attack_strength', 'defense', 'move_strength', 'harvest_strength',
        'temperment',
        'piece_type', '
        'food', 'diet', 'food_capacity',
        'harvest_goal', 'settle_progress', 'settle_threshold',
    ]

    def __post_init__(self):
        """Apply default stats from piece type if still at dataclass defaults."""
        defs = PIECE_DEFAULTS.get(self.piece_type)
        if defs is None:
            return
        atk, dfn, hp, diet, cap, harv, spd = defs

        # Only override if still at generic defaults
        if self.attack_strength == 1.0 and self.defense == 1.0:
            self.attack_strength  = atk
            self.defense          = dfn
            self.max_health       = hp
            self.health           = hp
            self.diet             = diet
            self.food_capacity    = cap
            self.harvest_strength = harv
            self.move_strength    = spd
            self.food             = cap  # start full

    def encode(self) -> str:
        """Encode piece as a single tab-delimited line."""
        vals = []
        for f in self.PIECE_FIELDS:
            if f == 'rules':
                v = ','.join(str(x) for x in self.instructions.rules)
            elif f == 'cursor':
                v = str(self.instructions.cursor)
            elif f == 'patrol':
                v = '1' if self.instructions.patrol else '0'
            else:
                v = getattr(self, f)
                if v is None:              v = ''
                elif isinstance(v, list):  v = ','.join(str(x) for x in v)
                elif isinstance(v, bool):  v = '1' if v else '0'
                elif isinstance(v, Enum):  v = v.value
                else:                      v = str(v)
            vals.append(str(v))
        return '\t'.join(vals)

    @classmethod
    def decode(cls, line: str) -> 'Piece':
        """Decode a tab-delimited line into a Piece."""
        parts = line.split('\t')
        kw = dict(zip(cls.PIECE_FIELDS, parts))

        # Ints
        for f in ['owner_id', 'size', 'health', 'max_health', 'sight',
                  'movement_range', 'current_position',
                  'facing', 'birth_year', 'squad','temperment'
                  'move_strength',
                  'settle_progress', 'settle_threshold']:
            kw[f] = int(kw[f]) if kw[f] else 0

        # Floats
        for f in ['food', 'diet', 'food_capacity',
                  'attack_strength', 'defense', 'harvest_strength']:
            kw[f] = float(kw[f]) if kw[f] else 0.0
        if kw['diet'] == 0.0: kw['diet'] = 1.0
        if kw['food_capacity'] == 0.0: kw['food_capacity'] = 10.0
        if kw['defense'] == 0.0: kw['defense'] = 1.0

        # Optional ints
        kw['location'] = int(kw['location']) if kw['location'] else None

        # Optional str
        kw['settlement_id'] = kw['settlement_id'] or None
        kw['name'] = kw['name'] or "Marvin"

        # Enums
        kw['piece_type'] = PieceType(kw['piece_type']) if kw['piece_type'] else PieceType.PAWN
        kw['harvest_goal'] = Resources(kw['harvest_goal']) if kw['harvest_goal'] else None

        # Build InstructionList from the three flat fields
        rules_str = kw.pop('rules', '')
        cursor = int(kw.pop('cursor', '0') or '0')
        patrol = kw.pop('patrol', '1') not in ('0', '')
        rules = [int(x) for x in rules_str.split(',') if x] if rules_str else []
        kw['instructions'] = InstructionList(rules, cursor, patrol)

        return cls(**kw)
    ```

Thanks for the fix. I am going to have it be eveyone in the cone. I am giving us flamethrowers.
I guess the question is how many rounds/how lethal I want combat to be. A queen should be able to take out a bunch of pawns, but maybe it takes a few knights to get a queen?

Should we build an pyomo or other parameter turner to figure this out?

This was the older simulator. what should I put into a cell so we can execute in this notebook

```python

# AUTOGENERATED! DO NOT EDIT! File to edit: ../../nbs/game/07_combat.ipynb

"""
Combat balancing via Pyomo optimization.

Simulates fixed-position battles to tune offense/defense stats such that:
  - Rook = high defense, low offense (tank)
  - Knight = high offense, low defense (glass cannon)  
  - Queen = balanced powerhouse
  - Target: 1 Queen ≈ 1 Rook + 2 Knights

The model runs turn-based combat to equilibrium and checks who survives.
"""

from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional
import random

# Optional Pyomo import — graceful fallback for environments without it
try:
    import pyomo.environ as pyo
    HAS_PYOMO = True
except ImportError:
    pyo = None
    HAS_PYOMO = False


# ════════════════════════════════════════════════════════════════════════════
# Combat Stats — the knobs we want to balance
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class CombatStats:
    """Offense/defense profile for a piece type."""
    offense: float = 1.0   # damage dealt per attack
    defense: float = 1.0   # damage reduction multiplier (higher = tougher)
    health: float = 100.0  # starting HP
    attack_range: int = 1  # hexes (not used in abstract sim, but for reference)
    
    @property
    def effective_hp(self) -> float:
        """HP adjusted for defense: how much raw damage needed to kill."""
        return self.health * self.defense


# Default profiles — Rook tanky, Knight glasscannon, Queen strong all-round
# Tuned for 1 Queen ≈ 1 Rook + 2 Knights balance
#
# Balance results (with 15% damage variance):
#   Q vs R+2K: ~25-60% Queen wins (sensitive to exact values)
#   Q vs R:    100% Queen wins
#   Q vs 2K:   100% Queen wins  
#   R vs 2K:   0% Rook wins (Knights punish tanks)
#   R vs K:    100% Rook wins
#   Q vs 2R:   0% Queen wins
#
# Design philosophy:
#   - Rook: high defense (1.6), high HP (130), low offense (2.5) = tank
#   - Knight: high offense (7.5), low defense (0.5), low HP (55) = glass cannon
#   - Queen: balanced high stats, slightly weaker than R+2K combined
#
COMBAT_PROFILES = {
    'QUEEN':  CombatStats(offense=8.4, defense=1.29, health=121),  # powerhouse (~59% vs R+2K)
    'ROOK':   CombatStats(offense=2.5, defense=1.6, health=130),   # tanky, low offense
    'KNIGHT': CombatStats(offense=7.5, defense=0.5, health=55),    # glass cannon, high burst
    'BISHOP': CombatStats(offense=4.0, defense=1.0, health=100),
    'PAWN':   CombatStats(offense=1.5, defense=0.8, health=80),
    'KING':   CombatStats(offense=3.0, defense=1.5, health=100),
}


# ════════════════════════════════════════════════════════════════════════════
# Combat Unit — a piece in the battle
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class CombatUnit:
    """A combatant in the abstract battle simulation."""
    id: str
    team: int  # 0 or 1
    piece_type: str
    stats: CombatStats
    hp: float = field(init=False)
    
    def __post_init__(self):
        self.hp = self.stats.health
    
    @property
    def alive(self) -> bool:
        return self.hp > 0
    
    def take_damage(self, raw_damage: float) -> float:
        """Apply damage after defense. Returns actual damage taken."""
        actual = raw_damage / self.stats.defense
        self.hp = max(0, self.hp - actual)
        return actual
    
    def attack(self, target: 'CombatUnit') -> float:
        """Attack target, return damage dealt."""
        if not self.alive or not target.alive:
            return 0.0
        return target.take_damage(self.stats.offense)


# ════════════════════════════════════════════════════════════════════════════
# Battle Simulator — turn-based combat to conclusion
# ════════════════════════════════════════════════════════════════════════════

@dataclass 
class BattleResult:
    """Outcome of a battle simulation."""
    winner: Optional[int]  # 0, 1, or None (draw)
    turns: int
    team0_survivors: int
    team1_survivors: int
    team0_total_hp: float
    team1_total_hp: float
    
    @property
    def margin(self) -> float:
        """Positive = team0 won by more, negative = team1 won by more."""
        return self.team0_total_hp - self.team1_total_hp


def run_battle(
    team0: list[CombatUnit],
    team1: list[CombatUnit],
    max_turns: int = 100,
    target_priority: str = 'lowest_hp',  # or 'random', 'highest_offense'
    damage_variance: float = 0.0  # 0.2 = ±20% damage variance
) -> BattleResult:
    """
    Simulate turn-based combat between two teams.
    
    Each turn:
      1. All units on team0 attack (simultaneously)
      2. All units on team1 attack (simultaneously)
      3. Check for victory
    
    Target selection: each attacker picks a living enemy.
    damage_variance: adds randomness to damage (0.2 = ±20%)
    """
    for unit in team0 + team1:
        unit.hp = unit.stats.health  # reset HP
    
    def pick_target(enemies: list[CombatUnit]) -> Optional[CombatUnit]:
        living = [e for e in enemies if e.alive]
        if not living:
            return None
        if target_priority == 'lowest_hp':
            return min(living, key=lambda u: u.hp)
        elif target_priority == 'highest_offense':
            return max(living, key=lambda u: u.stats.offense)
        else:
            return random.choice(living)
    
    def do_attack(attacker: CombatUnit, target: CombatUnit):
        if not attacker.alive or not target.alive:
            return
        base_dmg = attacker.stats.offense
        if damage_variance > 0:
            mult = 1.0 + random.uniform(-damage_variance, damage_variance)
            base_dmg *= mult
        target.take_damage(base_dmg)
    
    for turn in range(1, max_turns + 1):
        # Team 0 attacks
        for attacker in team0:
            if not attacker.alive:
                continue
            target = pick_target(team1)
            if target:
                do_attack(attacker, target)
        
        # Team 1 attacks
        for attacker in team1:
            if not attacker.alive:
                continue
            target = pick_target(team0)
            if target:
                do_attack(attacker, target)
        
        # Check victory
        t0_alive = [u for u in team0 if u.alive]
        t1_alive = [u for u in team1 if u.alive]
        
        if not t0_alive and not t1_alive:
            return BattleResult(None, turn, 0, 0, 0.0, 0.0)
        if not t1_alive:
            return BattleResult(0, turn, len(t0_alive), 0,
                              sum(u.hp for u in t0_alive), 0.0)
        if not t0_alive:
            return BattleResult(1, turn, 0, len(t1_alive),
                              0.0, sum(u.hp for u in t1_alive))
    
    # Timeout — whoever has more HP wins
    t0_hp = sum(u.hp for u in team0 if u.alive)
    t1_hp = sum(u.hp for u in team1 if u.alive)
    t0_alive = len([u for u in team0 if u.alive])
    t1_alive = len([u for u in team1 if u.alive])
    
    if t0_hp > t1_hp:
        winner = 0
    elif t1_hp > t0_hp:
        winner = 1
    else:
        winner = None
    
    return BattleResult(winner, max_turns, t0_alive, t1_alive, t0_hp, t1_hp)


# ════════════════════════════════════════════════════════════════════════════
# Scenario Builder — create standard matchups
# ════════════════════════════════════════════════════════════════════════════

def make_unit(piece_type: str, team: int, idx: int = 0, 
              profiles: dict = None) -> CombatUnit:
    """Create a combat unit from piece type string."""
    profiles = profiles or COMBAT_PROFILES
    stats = profiles.get(piece_type.upper(), COMBAT_PROFILES['PAWN'])
    return CombatUnit(
        id=f"{piece_type}_{team}_{idx}",
        team=team,
        piece_type=piece_type.upper(),
        stats=stats
    )


def scenario_queen_vs_rook_knights(profiles: dict = None) -> tuple[list, list]:
    """
    The key balancing scenario: 1 Queen vs 1 Rook + 2 Knights.
    Target: should be roughly even (queen wins ~50% or slight edge).
    """
    profiles = profiles or COMBAT_PROFILES
    team0 = [make_unit('QUEEN', 0, 0, profiles)]
    team1 = [
        make_unit('ROOK', 1, 0, profiles),
        make_unit('KNIGHT', 1, 0, profiles),
        make_unit('KNIGHT', 1, 1, profiles),
    ]
    return team0, team1


def scenario_mirror(piece_type: str, count: int = 1, 
                    profiles: dict = None) -> tuple[list, list]:
    """Mirror match — same composition on both sides."""
    profiles = profiles or COMBAT_PROFILES
    team0 = [make_unit(piece_type, 0, i, profiles) for i in range(count)]
    team1 = [make_unit(piece_type, 1, i, profiles) for i in range(count)]
    return team0, team1


def scenario_custom(team0_spec: list[str], team1_spec: list[str],
                   profiles: dict = None) -> tuple[list, list]:
    """
    Custom scenario from piece type lists.
    
    Example: scenario_custom(['QUEEN', 'PAWN'], ['ROOK', 'ROOK', 'KNIGHT'])
    """
    profiles = profiles or COMBAT_PROFILES
    team0 = [make_unit(pt, 0, i, profiles) for i, pt in enumerate(team0_spec)]
    team1 = [make_unit(pt, 1, i, profiles) for i, pt in enumerate(team1_spec)]
    return team0, team1


# ════════════════════════════════════════════════════════════════════════════
# Batch Testing — run scenarios multiple times for stability
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class ScenarioSummary:
    """Aggregated results from running a scenario multiple times."""
    name: str
    runs: int
    team0_wins: int
    team1_wins: int
    draws: int
    avg_turns: float
    avg_margin: float  # positive = team0 advantage
    
    @property
    def team0_winrate(self) -> float:
        return self.team0_wins / self.runs if self.runs > 0 else 0.0
    
    @property
    def team1_winrate(self) -> float:
        return self.team1_wins / self.runs if self.runs > 0 else 0.0
    
    def __repr__(self):
        return (f"ScenarioSummary({self.name}: "
                f"T0={self.team0_winrate:.1%}, T1={self.team1_winrate:.1%}, "
                f"draws={self.draws}, avg_turns={self.avg_turns:.1f})")


def run_scenario_batch(
    team0_spec: list[str],
    team1_spec: list[str],
    runs: int = 100,
    profiles: dict = None,
    name: str = None,
    damage_variance: float = 0.15  # default 15% variance for interesting outcomes
) -> ScenarioSummary:
    """Run a scenario multiple times and aggregate results."""
    profiles = profiles or COMBAT_PROFILES
    name = name or f"{'+'.join(team0_spec)} vs {'+'.join(team1_spec)}"
    
    results = []
    for _ in range(runs):
        team0, team1 = scenario_custom(team0_spec, team1_spec, profiles)
        result = run_battle(team0, team1, target_priority='lowest_hp', 
                           damage_variance=damage_variance)
        results.append(result)
    
    t0_wins = sum(1 for r in results if r.winner == 0)
    t1_wins = sum(1 for r in results if r.winner == 1)
    draws = sum(1 for r in results if r.winner is None)
    avg_turns = sum(r.turns for r in results) / len(results)
    avg_margin = sum(r.margin for r in results) / len(results)
    
    return ScenarioSummary(name, runs, t0_wins, t1_wins, draws, avg_turns, avg_margin)


# ════════════════════════════════════════════════════════════════════════════
# Pyomo Balance Optimizer — find stats that achieve target win rates
# ════════════════════════════════════════════════════════════════════════════

def evaluate_balance(
    queen_offense: float, queen_defense: float, queen_health: float,
    rook_offense: float, rook_defense: float, rook_health: float,
    knight_offense: float, knight_defense: float, knight_health: float,
    runs: int = 50
) -> dict:
    """
    Evaluate a set of combat stats by running key scenarios.
    
    Returns metrics including the key "queen_vs_rk_winrate" which should be ~0.5.
    """
    profiles = {
        'QUEEN':  CombatStats(queen_offense, queen_defense, queen_health),
        'ROOK':   CombatStats(rook_offense, rook_defense, rook_health),
        'KNIGHT': CombatStats(knight_offense, knight_defense, knight_health),
        'BISHOP': COMBAT_PROFILES['BISHOP'],
        'PAWN':   COMBAT_PROFILES['PAWN'],
        'KING':   COMBAT_PROFILES['KING'],
    }
    
    # Key scenario: Queen vs Rook + 2 Knights
    qvrk = run_scenario_batch(['QUEEN'], ['ROOK', 'KNIGHT', 'KNIGHT'], 
                              runs=runs, profiles=profiles, name='Q_vs_R2K')
    
    # Sanity checks
    q_vs_r = run_scenario_batch(['QUEEN'], ['ROOK'], runs=runs//2, profiles=profiles)
    q_vs_2k = run_scenario_batch(['QUEEN'], ['KNIGHT', 'KNIGHT'], runs=runs//2, profiles=profiles)
    r_vs_2k = run_scenario_batch(['ROOK'], ['KNIGHT', 'KNIGHT'], runs=runs//2, profiles=profiles)
    
    return {
        'queen_vs_rk_winrate': qvrk.team0_winrate,
        'queen_vs_rk_margin': qvrk.avg_margin,
        'queen_vs_rook': q_vs_r.team0_winrate,
        'queen_vs_2knight': q_vs_2k.team0_winrate,
        'rook_vs_2knight': r_vs_2k.team0_winrate,
        'profiles': profiles,
    }


def balance_score(metrics: dict, target_qvrk: float = 0.5) -> float:
    """
    Score a balance configuration. Lower is better.
    
    Primary goal: queen_vs_rk_winrate ≈ target (default 50%)
    Secondary: queen should beat rook 1v1, queen should beat 2 knights
    """
    qvrk_error = abs(metrics['queen_vs_rk_winrate'] - target_qvrk)
    
    # Penalties for other imbalances
    penalties = 0.0
    
    # Queen should beat single rook (say 70%+)
    if metrics['queen_vs_rook'] < 0.7:
        penalties += (0.7 - metrics['queen_vs_rook']) * 0.5
    
    # Queen should beat 2 knights (say 60%+)
    if metrics['queen_vs_2knight'] < 0.6:
        penalties += (0.6 - metrics['queen_vs_2knight']) * 0.3
    
    # Rook should struggle vs 2 knights (glasscannons should punish tank)
    # Target: rook wins ~40%
    rook_2k_target = 0.4
    rook_2k_error = abs(metrics['rook_vs_2knight'] - rook_2k_target) * 0.2
    
    return qvrk_error + penalties + rook_2k_error


# ════════════════════════════════════════════════════════════════════════════
# Grid Search — simple parameter sweep (Pyomo overkill for discrete sim)
# ════════════════════════════════════════════════════════════════════════════

def grid_search_balance(
    queen_offense_range: tuple = (5.0, 9.0, 1.0),
    queen_defense_range: tuple = (1.0, 1.6, 0.2),
    rook_offense_range: tuple = (2.0, 5.0, 1.0),
    rook_defense_range: tuple = (1.5, 2.5, 0.25),
    knight_offense_range: tuple = (5.0, 8.0, 1.0),
    knight_defense_range: tuple = (0.5, 0.9, 0.1),
    runs_per_eval: int = 30,
    verbose: bool = True
) -> dict:
    """
    Grid search over stat combinations to find balanced values.
    
    Each range is (min, max, step).
    Returns best configuration found.
    """
    import numpy as np
    
    def frange(start, stop, step):
        return list(np.arange(start, stop + step/2, step))
    
    best_score = float('inf')
    best_config = None
    best_metrics = None
    
    q_offs = frange(*queen_offense_range)
    q_defs = frange(*queen_defense_range)
    r_offs = frange(*rook_offense_range)
    r_defs = frange(*rook_defense_range)
    k_offs = frange(*knight_offense_range)
    k_defs = frange(*knight_defense_range)
    
    total = len(q_offs) * len(q_defs) * len(r_offs) * len(r_defs) * len(k_offs) * len(k_defs)
    if verbose:
        print(f"Grid search: {total} combinations")
    
    # Fixed health values (can extend to search these too)
    q_hp, r_hp, k_hp = 120, 150, 70
    
    count = 0
    for qo in q_offs:
        for qd in q_defs:
            for ro in r_offs:
                for rd in r_defs:
                    for ko in k_offs:
                        for kd in k_defs:
                            count += 1
                            metrics = evaluate_balance(
                                qo, qd, q_hp,
                                ro, rd, r_hp,
                                ko, kd, k_hp,
                                runs=runs_per_eval
                            )
                            score = balance_score(metrics)
                            
                            if score < best_score:
                                best_score = score
                                best_config = {
                                    'queen': (qo, qd, q_hp),
                                    'rook': (ro, rd, r_hp),
                                    'knight': (ko, kd, k_hp),
                                }
                                best_metrics = metrics
                                if verbose:
                                    print(f"  [{count}/{total}] New best (score={score:.3f}): "
                                          f"Q={qo}/{qd}, R={ro}/{rd}, K={ko}/{kd} "
                                          f"→ Q_vs_R2K={metrics['queen_vs_rk_winrate']:.1%}")
    
    return {
        'score': best_score,
        'config': best_config,
        'metrics': best_metrics,
    }


# ════════════════════════════════════════════════════════════════════════════
# Convenience: Quick Test
# ════════════════════════════════════════════════════════════════════════════

def test_default_balance(runs: int = 100, verbose: bool = True) -> dict:
    """Test the default COMBAT_PROFILES and report results."""
    profiles = COMBAT_PROFILES
    
    scenarios = [
        (['QUEEN'], ['ROOK', 'KNIGHT', 'KNIGHT'], 'Q vs R+2K (target: ~50%)'),
        (['QUEEN'], ['ROOK'], 'Q vs R (Queen should win)'),
        (['QUEEN'], ['KNIGHT', 'KNIGHT'], 'Q vs 2K (Queen should win)'),
        (['ROOK'], ['KNIGHT', 'KNIGHT'], 'R vs 2K (Knights should win)'),
        (['ROOK'], ['KNIGHT'], 'R vs K (Rook should win)'),
        (['QUEEN'], ['ROOK', 'ROOK'], 'Q vs 2R (Rooks should win)'),
    ]
    
    results = {}
    for t0, t1, name in scenarios:
        summary = run_scenario_batch(t0, t1, runs=runs, profiles=profiles, name=name)
        results[name] = summary
        if verbose:
            print(f"{name}: Team0 wins {summary.team0_winrate:.1%}, "
                  f"avg {summary.avg_turns:.1f} turns")
    
    return results


# ════════════════════════════════════════════════════════════════════════════
# Main — run if executed directly
# ════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    print("=== Combat Balance Test ===\n")
    print("Default profiles:")
    for name, stats in COMBAT_PROFILES.items():
        print(f"  {name}: off={stats.offense}, def={stats.defense}, hp={stats.health}")
    print()
    
    test_default_balance(runs=100)

```

Does it look like I put it in ok? should we build a demo

In [ ]:
# ── Combat Demo: Queen + 2 Pawns vs 3 Knights ──
myStuff = GameParts()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
grid = myStuff.grid; elevs = myStuff.terr.elevations
food_tiers = fy.tiers; countries = myStuff.terr.fields.get('country')

# Pick a central hex with some room
center = myStuff.queen.location
ring1 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(1)]
ring2 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(2)]
# Filter valid hexes
ring1 = [h for h in ring1 if 0 <= h < len(elevs) and elevs[h] >= 1]
ring2 = [h for h in ring2 if 0 <= h < len(elevs) and elevs[h] >= 1]

# ── Team 0: Queen + 2 Pawns (defenders) ──
queen = Piece(name="WarQueen", piece_type=PieceType.QUEEN, owner_id=0,
              location=center, facing=4, temperment=20,
              instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

p1 = Piece(name="Shield1", piece_type=PieceType.PAWN, owner_id=0,
           location=ring1[0], facing=4, temperment=40,
           instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

p2 = Piece(name="Shield2", piece_type=PieceType.PAWN, owner_id=0,
           location=ring1[1] if len(ring1) > 1 else ring1[0], facing=4, temperment=40,
           instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

# ── Team 1: 3 Knights (attackers) ──
k1 = Piece(name="Raider1", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

k2 = Piece(name="Raider2", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[1] if len(ring2) > 1 else ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

k3 = Piece(name="Raider3", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[2] if len(ring2) > 2 else ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

all_pieces = [queen, p1, p2, k1, k2, k3]

for p in all_pieces:
    p.food = 999
    p.food_capacity = 999

# ── Run ──
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=all_pieces, countries=countries,
)
sim.run(num_turns=20)
sim.summary()

# ── Show combat events ──
df = sim.df
combat_df = df[df.event.isin(['attacked', 'countered', 'died'])]
print(f"\n{len(combat_df)} combat events:")
print(combat_df[['turn', 'event', 'piece_name', 'dst_piece_id', 'amount']].to_string(index=False))


In [ ]:
show(k1)

In [ ]:
show(queen)

So I think the queen starves

So I wonder about placing a knight in the queens blind spot

In [ ]:
# ── Blind Spot Demo: Knight behind the Queen ──
center = myStuff.queen.location
ring1 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(1)]
ring1 = [h for h in ring1 if 0 <= h < len(elevs) and elevs[h] >= 1]

# Queen faces direction 4 (roughly east)
queen2 = Piece(name="BlindQueen", piece_type=PieceType.QUEEN, owner_id=0,
               location=center, facing=4, temperment=20,
               instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

# Knight placed BEHIND the queen — direction 1 is opposite of 4
behind_hex = grid.hexposition_to_index(HexPosition.directions()[1], center)
sneaky = Piece(name="Sneaky", piece_type=PieceType.KNIGHT, owner_id=1,
               location=behind_hex, facing=4, temperment=0,  # always attacks
               instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

# Check: is the knight in the queen's sight cone?
queen_vis = set(queen2.hexes_in_sight(grid, elevs, elevation_mult=0.005, facing_only=True))
knight_vis = set(sneaky.hexes_in_sight(grid, elevs, elevation_mult=0.005, facing_only=True))
print(f"Knight at {behind_hex} in queen's cone? {behind_hex in queen_vis}")
print(f"Queen at {center} in knight's cone? {center in knight_vis}")

for p in [queen2, sneaky]:
    p.food = 999; p.food_capacity = 999

sim2 = FoodSimulator(grid=grid, elevations=elevs, food_tiers=food_tiers,
                     pieces=[queen2, sneaky], countries=countries)
sim2.run(num_turns=30)
sim2.summary()

df2 = sim2.df
combat2 = df2[df2.event.isin(['attacked', 'countered', 'died'])]
print(f"\n{len(combat2)} combat events:")
print(combat2[['turn', 'event', 'piece_name', 'dst_piece_id', 'amount']].to_string(index=False))


So what would combat phase look like if you can only counter people in your vision